# Image Editing LLM Pipeline — Phase 3: Inference (v3.0, grounding-model)

**v3.0 vs v2.2** — replaces VLM-bbox-only localization with a GroundingDINO
detector at inference time. The Phase-1 fine-tuned VLM is still used for
(a) producing the structured JSON edit prediction (edit_type, edit_description,
subject phrase) and (b) producing the hidden-state vector that conditions the
SD bridge — so the Phase 2 projection adapter and UNet LoRA continue to work
without retraining.

### Why this change
On the held-out samples from v2.2, two failure modes dominated:
  * **Degenerate VLM bbox** (e.g. `[0,0,1000,1000]` for "make the motorcycle red")
    causes SAM2 to fragment over the whole image because there is no useful
    spatial signal in the box prompt.
  * **Under-localized VLM bbox** (e.g. covering only "Merry" but not "Christmas")
    causes SAM2 to faithfully segment a partial object — the inpainter then
    edits only half of the intended target.

GroundingDINO is a pre-trained open-vocabulary detector that takes a free-form
text phrase ("the motorcycle", "the bread", "the text on the cake") and returns
calibrated bounding boxes. Routing localization through it sidesteps the
under-trained VLM bbox head while keeping the trained conditioning pipeline.

### New components in v3.0
  * §22.5 — Load `IDEA-Research/grounding-dino-tiny` (HF Transformers, no extra install).
  * §25.4 — `extract_subject_phrase()` derives the noun phrase from VLM JSON.
  * §26.1 — `bbox_to_sam2_mask` rewritten:
      - Run GroundingDINO with the subject phrase → candidate bboxes.
      - Sanity-filter VLM bbox (degenerate / over-large detection).
      - Pick best bbox via priority chain (grounding → VLM → full-image fallback).
      - SAM2 with `multimask_output=True` + box and centre-point prompts.
      - CLIP image-text rerank over candidate masks.
      - Morphological closing + largest-connected-component cleanup.

### Pipeline contract — UNCHANGED
  * VLM hidden state extraction (§25.2) is byte-identical to v2.2 / Phase 2.
  * `build_combined_conditioning` (§23.3) is byte-identical to Phase 2 §16.
  * 9-channel UNet input, CFG handling, scheduler parity asserts — UNCHANGED.

### Spec compliance notes
  * Phase 2 still trains on RLE ground-truth masks. Phase 3 still uses
    inference-derived masks. The two paths remain strictly separated.
  * Train/inference conditioning parity is preserved: no conditioning input
    has been re-routed; only the *spatial mask* derivation changed.
  * The grounding model is treated as an external pre-trained component in
    the same role SAM2 occupies: a frozen mask helper at inference, never
    in the training loop.

## §0 — Global Configuration
*Identical base to Phase 0/1/2; Phase 3 fields appended. Edit this cell only.*

In [1]:
# ── §0.1  GLOBAL CONFIG — edit this cell only ──────────────────────────────
# v3.0 additions are marked with: # ← v3.0 (grounding model + mask rerank)

import os, sys, json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Tuple

@dataclass
class PipelineConfig:
    # ── Drive root ────────────────────────────────────────────────────────
    drive_root: Path = Path("/content/drive/MyDrive/img_edit_pipeline")

    # ── Dataset ───────────────────────────────────────────────────────────
    hf_dataset_id: str         = "sysuyy/ImgEdit"
    hf_configs: Optional[list] = None
    n_subset: int              = 10_000

    expected_edit_types: tuple = (
        "action", "add", "adjust", "background", "content",
        "hybrid", "reference", "remove", "replace", "style", "version",
    )
    mask_dataset_id: str = "sysuyy/ImgEdit_recap_mask"

    global_adjust_keywords: tuple = (
        "lighting", "light", "brightness", "exposure", "contrast",
        "saturation", "hue", "tone", "tones", "overall", "scene",
        "atmosphere", "color temperature", "white balance",
    )

    # ── Phase 1 model ─────────────────────────────────────────────────────
    vlm_model_id: str       = "Qwen/Qwen2.5-VL-3B-Instruct"
    vlm_quant_bits: int     = 4
    vlm_lora_r: int         = 16
    vlm_lora_alpha: int     = 32
    vlm_lora_dropout: float = 0.05
    vlm_hidden_dim: int     = 2048   # Qwen2.5-VL-3B confirmed hidden size

    # ── Phase 2 model ─────────────────────────────────────────────────────
    sd_model_id: str        = "runwayml/stable-diffusion-inpainting"
    sd_cross_attn_dim: int  = 768    # SD 1.5 cross-attention dim (constant)
    sd_lora_r: int          = 8
    sd_lora_alpha: int      = 16
    sd_lora_dropout: float  = 0.0
    sd_lora_target_modules: tuple = ("to_q", "to_k", "to_v", "to_out.0")
    vae_scale_factor: float = 0.18215
    phase2_resolution: int  = 512

    # ── Phase 3 inference ─────────────────────────────────────────────────
    phase3_resolution: int          = 512
    phase3_num_inference_steps: int = 20
    phase3_guidance_scale: float    = 7.5
    phase3_max_new_tokens: int      = 256
    phase3_sam2_model_cfg: str      = "sam2.1/sam2.1_hiera_l"
    phase3_mixed_precision: str     = "fp16"
    phase3_fallback_bbox: tuple     = (0, 0, 1000, 1000)

    # ── v3.0: Grounding model (open-vocabulary detection at inference) ─────
    # IDEA-Research/grounding-dino-tiny ships in transformers >= 4.40.
    # No extra pip install required; loaded via AutoProcessor +
    # AutoModelForZeroShotObjectDetection.
    grounding_model_id: str         = "IDEA-Research/grounding-dino-tiny"  # ← v3.0
    grounding_box_threshold: float  = 0.25   # ← v3.0  (per-box conf threshold)
    grounding_text_threshold: float = 0.20   # ← v3.0  (per-token conf threshold)
    grounding_max_boxes: int        = 5      # ← v3.0  (cap before union/rerank)
    # When VLM bbox area > this fraction of the image, treat as degenerate
    # and prefer grounding instead. Catches the [0,0,1000,1000] failure mode.
    bbox_degenerate_max_frac: float = 0.7    # ← v3.0
    # When VLM bbox area < this fraction of the image, treat as too-small
    # and prefer grounding instead.
    bbox_degenerate_min_frac: float = 0.001  # ← v3.0

    # ── v3.0: SAM2 multi-mask + CLIP rerank ───────────────────────────────
    # multimask_output=True returns 3 candidate masks; we pick the best via
    # a CLIP image-text similarity rerank against the subject phrase.
    sam2_multimask_output: bool     = True   # ← v3.0
    sam2_use_centre_point: bool     = True   # ← v3.0  (add centre-of-bbox positive point)
    sam2_clip_rerank: bool          = True   # ← v3.0
    # Mask post-processing parameters
    mask_morph_close_kernel: int    = 7      # ← v3.0  (px, used on full-res mask)
    mask_min_component_area_frac: float = 0.002  # ← v3.0  (drop CCs smaller than this)
    mask_keep_largest_component: bool   = True   # ← v3.0
    # When the global-edit short-circuit fires (bbox_area >= this), skip SAM2
    # and emit a full-image mask. This is the same global-edit guard as v2.2.
    global_edit_area_frac: float    = 0.90   # ← v3.0 (was hard-coded 0.90 in v2.2)

    # ── Training (kept for shared CFG structure) ───────────────────────────
    phase1_epochs: int         = 3
    phase1_batch_size: int     = 2
    phase1_grad_accum: int     = 8
    phase1_lr: float           = 2e-4
    phase1_max_seq_len: int    = 2560
    phase1_max_img_pixels: int = 802_816
    phase1_warmup_steps: int   = 100
    phase1_eval_steps: int     = 200
    phase2_epochs: int         = 5
    phase2_batch_size: int     = 1
    phase2_grad_accum: int     = 16
    phase2_lr: float           = 1e-4
    phase2_warmup_steps: int   = 200
    phase2_shard_size: int     = 5_000
    phase2_max_grad_norm: float = 1.0
    phase2_num_workers: int    = 2
    phase2_save_steps: int     = 500
    phase2_mixed_precision: str = "fp16"
    phase2_num_train_timesteps: int = 1000
    phase2_beta_schedule: str  = "linear"

    max_invalid_rle_frac: float = 0.05
    max_fallback_frac: float    = 0.15

    # ── Derived paths ─────────────────────────────────────────────────────
    @property
    def data_dir(self) -> Path:
        return self.drive_root / "data" / "imgedit_subset"

    @property
    def benchmark_dir(self) -> Path:
        return self.drive_root / "data" / "benchmark"

    @property
    def ckpt_phase1(self) -> Path:
        return self.drive_root / "checkpoints" / "phase1_vlm"

    @property
    def ckpt_phase2(self) -> Path:
        return self.drive_root / "checkpoints" / "phase2_diffusion_v2_r16"

    @property
    def ckpt_phase2_final(self) -> Path:
        return self.ckpt_phase2 / "final"

    @property
    def outputs_eval(self) -> Path:
        # v3.0 output directory is a sibling of v2.2 so old results are not overwritten.
        return self.drive_root / "outputs" / "eval_v3"   # ← v3.0

    @property
    def outputs_scores(self) -> Path:
        return self.drive_root / "outputs" / "scores"

    @property
    def manifest_path(self) -> Path:
        return self.data_dir / "samples.json"

    @property
    def filtered_manifest_path(self) -> Path:
        return self.data_dir / "samples_filtered.json"

    @property
    def hidden_states_dir(self) -> Path:
        return self.data_dir / "vlm_hidden_states"

    @property
    def audit_path(self) -> Path:
        return self.data_dir / "audit_report.json"

    @property
    def sam2_dir(self) -> Path:
        return self.drive_root / "sam2"

    @property
    def sam2_checkpoint(self) -> Path:
        return self.sam2_dir / "sam2.1_hiera_large.pt"


CFG = PipelineConfig()
print("Config loaded (v3.0).")
print(f"  Drive root              : {CFG.drive_root}")
print(f"  Phase 1 checkpoint      : {CFG.ckpt_phase1}")
print(f"  Phase 2 checkpoint      : {CFG.ckpt_phase2_final}")
print(f"  SAM2 checkpoint         : {CFG.sam2_checkpoint}")
print(f"  Grounding model         : {CFG.grounding_model_id}")
print(f"  SD model                : {CFG.sd_model_id}")
print(f"  VLM model               : {CFG.vlm_model_id}")
print(f"  Inference steps (DDIM)  : {CFG.phase3_num_inference_steps}")
print(f"  Guidance scale (CFG)    : {CFG.phase3_guidance_scale}")
print(f"  Phase3 resolution       : {CFG.phase3_resolution}")
print(f"  v3 outputs dir          : {CFG.outputs_eval}")


Config loaded (v3.0).
  Drive root              : /content/drive/MyDrive/img_edit_pipeline
  Phase 1 checkpoint      : /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm
  Phase 2 checkpoint      : /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final
  SAM2 checkpoint         : /content/drive/MyDrive/img_edit_pipeline/sam2/sam2.1_hiera_large.pt
  Grounding model         : IDEA-Research/grounding-dino-tiny
  SD model                : runwayml/stable-diffusion-inpainting
  VLM model               : Qwen/Qwen2.5-VL-3B-Instruct
  Inference steps (DDIM)  : 20
  Guidance scale (CFG)    : 7.5
  Phase3 resolution       : 512
  v3 outputs dir          : /content/drive/MyDrive/img_edit_pipeline/outputs/eval_v3


## §0.2 — Google Drive Mount & Directory Scaffold

In [2]:
# ── §0.2  Mount Drive and create Phase 3 directories ─────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

_dirs = [
    CFG.data_dir,
    CFG.ckpt_phase1,
    CFG.ckpt_phase2_final,
    CFG.outputs_eval,
    CFG.outputs_scores,
    CFG.sam2_dir,
]
for d in _dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  ok  {d}")
print("Drive mounted and directories verified.")

Mounted at /content/drive
  ok  /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset
  ok  /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm
  ok  /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final
  ok  /content/drive/MyDrive/img_edit_pipeline/outputs/eval_v3
  ok  /content/drive/MyDrive/img_edit_pipeline/outputs/scores
  ok  /content/drive/MyDrive/img_edit_pipeline/sam2
Drive mounted and directories verified.


## §1.1 — Package Installation
⚠ **Run once per runtime.** After this cell, restart the kernel then continue from §1.2.

Same pinned versions as Phase 2 plus SAM2 installation from GitHub (required for Phase 3 mask generation).

In [3]:
# ── §1.1  Install pinned dependencies ────────────────────────────────────
# After this cell finishes: RESTART KERNEL, then run §1.2.

import subprocess, sys

def _install(pkg, label=None):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    tag = label or pkg
    if r.returncode == 0:
        print(f"  ok  {tag}")
    else:
        print(f"  FAILED  {tag}")
        print(r.stderr[-400:])

# ── Step 1: numpy<2 FIRST ─────────────────────────────────────────────────
print("Step 1: numpy<2 (must come before all other installs)...")
_install("numpy<2", "numpy<2")

# ── Step 2: Core packages ─────────────────────────────────────────────────
print("\nStep 2: Phase 3 packages...")
for pkg, label in [
    ("diffusers>=0.27.0",     "diffusers>=0.27"),
    ("peft>=0.10.0",          "peft>=0.10"),
    ("accelerate>=0.28.0",    "accelerate>=0.28"),
    ("transformers>=4.49.0",  "transformers>=4.49"),
    ("bitsandbytes>=0.44.0",  "bitsandbytes>=0.44"),
    ("pycocotools>=2.0.7",    "pycocotools>=2.0.7"),
    ("tqdm>=4.66",            "tqdm>=4.66"),
    ("Pillow>=10.0",          "Pillow>=10"),
    ("qwen-vl-utils>=0.0.8",  "qwen-vl-utils>=0.0.8"),
]:
    _install(pkg, label)

# ── Step 3: torchao fix (PEFT LoRA on Colab) ─────────────────────────────
print("\nStep 3: torchao>=0.16.0 (PEFT LoRA compatibility fix)...")
_install("torchao>=0.16.0", "torchao>=0.16.0")

# ── Step 4: SAM2 from GitHub ─────────────────────────────────────────────
# SAM2 is not on PyPI — must be installed from source.
# We clone to /content/sam2 (Colab local, lost on restart).
# The Drive-cached checkpoint (CFG.sam2_checkpoint) persists across restarts.
#
# SAM2 install fragility notes:
#   - torch>=2.5.1 required (spec §1.2 REQUIRED dict confirms this)
#   - The `sam2` package installs Hydra for config resolution
#   - After kernel restart, re-run only §21.1 (SAM2 re-install) — checkpoint
#     is loaded from Drive so no re-download needed.
print("\nStep 4: SAM2 from GitHub...")
_sam2_dir = "/content/sam2"
import os
if not os.path.exists(_sam2_dir):
    r = subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/facebookresearch/sam2.git", _sam2_dir],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"  FAILED: git clone sam2: {r.stderr[-300:]}")
    else:
        print("  ok  git clone sam2")
else:
    print("  ok  /content/sam2 already exists")

_r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", _sam2_dir],
    capture_output=True, text=True
)
if _r.returncode == 0:
    print("  ok  sam2 installed from source")
else:
    print(f"  FAILED: pip install sam2: {_r.stderr[-300:]}")

print()
print("Installation complete.")
print("⚠  RESTART KERNEL NOW, then run §1.2.")
print("   Runtime → Restart session  (Ctrl+M .)  then continue from §1.2.")

Step 1: numpy<2 (must come before all other installs)...
  ok  numpy<2

Step 2: Phase 3 packages...
  ok  diffusers>=0.27
  ok  peft>=0.10
  ok  accelerate>=0.28
  ok  transformers>=4.49
  ok  bitsandbytes>=0.44
  ok  pycocotools>=2.0.7
  ok  tqdm>=4.66
  ok  Pillow>=10
  ok  qwen-vl-utils>=0.0.8

Step 3: torchao>=0.16.0 (PEFT LoRA compatibility fix)...
  ok  torchao>=0.16.0

Step 4: SAM2 from GitHub...
  ok  git clone sam2
  ok  sam2 installed from source

Installation complete.
⚠  RESTART KERNEL NOW, then run §1.2.
   Runtime → Restart session  (Ctrl+M .)  then continue from §1.2.


## §1.2 — Import & Version Sanity Checks
*Run after every kernel restart. All assertions must pass before any Phase 3 cell runs.*

In [2]:
# ── §1.2  Import and version gate ────────────────────────────────────────
import importlib, importlib.metadata, sys
from pathlib import Path

_failures = []

def _ver(pkg_name: str):
    try:
        return importlib.metadata.version(pkg_name)
    except importlib.metadata.PackageNotFoundError:
        return None

def _parse(ver_str: str) -> tuple:
    clean = ver_str.split("+")[0].split("-")[0]
    parts = clean.split(".")
    nums  = []
    for p in parts[:3]:
        try:    nums.append(int(p))
        except: nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def _check(pip_name, min_ver, max_ver=None, mod_name=None):
    ver = _ver(pip_name)
    if ver is None and mod_name:
        try:
            m = importlib.import_module(mod_name)
            ver = getattr(m, "__version__", None)
        except ImportError:
            pass
    if ver is None:
        _failures.append(f"  MISSING: {pip_name} — run §1.1 and restart kernel")
        return
    inst = _parse(ver)
    minv = _parse(min_ver)
    ok   = inst >= minv
    if max_ver:
        ok = ok and inst < _parse(max_ver)
    if ok:
        print(f"  ok  {pip_name}=={ver}")
    else:
        c = f">= {min_ver}" + (f", < {max_ver}" if max_ver else "")
        _failures.append(f"  WRONG: {pip_name}=={ver} (need {c})")

_check("numpy",        min_ver="1.0.0", max_ver="2.0.0")
_check("torch",        min_ver="2.5.1", mod_name="torch")
_check("diffusers",    min_ver="0.27.0")
_check("peft",         min_ver="0.10.0")
_check("accelerate",   min_ver="0.28.0")
# v3.0: GroundingDINO requires transformers >= 4.40. We already pin >= 4.49 for
# Qwen2.5-VL, so the grounding model class is guaranteed to be available.
_check("transformers", min_ver="4.49.0")
_check("bitsandbytes", min_ver="0.44.0")
_check("Pillow",       min_ver="10.0.0", mod_name="PIL")
_check("pycocotools",  min_ver="2.0.0")
# v3.0: scipy is used for morphological mask cleanup. It ships with Colab by default.
_check("scipy",        min_ver="1.10.0")

_tao_ver = _ver("torchao")
if _tao_ver is not None:
    _check("torchao", min_ver="0.16.0")
else:
    print("  ok  torchao (not installed — PEFT will use standard dispatcher)")

if _failures:
    raise RuntimeError(
        "Version gate failed:\n" + "\n".join(_failures) +
        "\n\nFixes:\n"
        "  numpy wrong  : pip install 'numpy<2' then RESTART KERNEL\n"
        "  pkg missing  : run §1.1, then RESTART KERNEL, then re-run §1.2\n"
        "  torch wrong  : Colab runtime must have torch>=2.5.1"
    )

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device found. "
        "Switch to Runtime → Change runtime type → GPU (A100 preferred)."
    )

_device_name = torch.cuda.get_device_name(0)
_vram_gb     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n  CUDA device  : {_device_name}")
print(f"  VRAM         : {_vram_gb:.1f} GB")
if _vram_gb < 22:
    print("  WARNING: <22 GB VRAM. v3.0 loads VLM + SAM2 + GroundingDINO + SD — "
          "recommend A100 (40GB).")

DEVICE = "cuda"
print(f"  DEVICE       : {DEVICE}")

# ── Qwen2.5-VL ────────────────────────────────────────────────────────────
try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    print("  ok  Qwen2_5_VLForConditionalGeneration importable")
except ImportError as e:
    raise EnvironmentError(f"Qwen2.5-VL not in transformers>=4.49.0: {e}")

# ── v3.0: GroundingDINO ──────────────────────────────────────────────────
# AutoModelForZeroShotObjectDetection + AutoProcessor cover the GroundingDINO
# class. We do NOT install the standalone groundingdino-py package: HF
# Transformers ships its own implementation that is faster to set up on Colab
# and avoids the CUDA-extension build that the original repo requires.
try:
    from transformers import AutoModelForZeroShotObjectDetection
    print("  ok  AutoModelForZeroShotObjectDetection importable (v3.0 grounding)")
except ImportError as e:
    raise EnvironmentError(
        f"GroundingDINO classes not found in transformers — "
        f"need transformers >= 4.40 (we pin >= 4.49). Error: {e}"
    )

# ── SAM2 ─────────────────────────────────────────────────────────────────
import sys as _sys_sam2, os as _os_sam2

_saved_cwd     = _os_sam2.getcwd()
_saved_syspath = _sys_sam2.path[:]
_SAM2_REPO_PARENT = '/content'
_sys_sam2.path = [
    p for p in _sys_sam2.path
    if p != ''
    and _os_sam2.path.normpath(p) != _os_sam2.path.normpath(_SAM2_REPO_PARENT)
]
for _k in [k for k in list(_sys_sam2.modules) if k == 'sam2' or k.startswith('sam2.')]:
    del _sys_sam2.modules[_k]
_os_sam2.chdir('/root')
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
finally:
    _os_sam2.chdir(_saved_cwd)
    _sys_sam2.path = _saved_syspath

try:
    _ = build_sam2
    print("  ok  sam2 importable (build_sam2, SAM2ImagePredictor)")
except NameError as e:
    raise EnvironmentError(
        f"SAM2 not importable after sys.path fix. Re-run §21.1. Error: {e}"
    )

# ── Diffusers components ───────────────────────────────────────────────────
from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
from peft import PeftModel, LoraConfig, get_peft_model
print("  ok  diffusers + peft components importable")

# ── pycocotools ───────────────────────────────────────────────────────────
import pycocotools.mask as mask_utils
print("  ok  pycocotools.mask importable")

# ── v3.0: scipy.ndimage for mask post-processing ─────────────────────────
from scipy import ndimage as _scipy_ndimage
print("  ok  scipy.ndimage importable (v3.0 mask post-processing)")

# ── CFG guard ─────────────────────────────────────────────────────────────
try:
    _ = CFG
    print("  ok  CFG in scope")
except NameError:
    raise RuntimeError("CFG not defined — re-run §0.1 config cell first.")

print()
print("§1.2 passed — all imports and versions OK (v3.0).")


  ok  numpy==1.26.4
  ok  torch==2.10.0+cu128
  ok  diffusers==0.37.1
  ok  peft==0.19.1
  ok  accelerate==1.13.0
  ok  transformers==5.0.0
  ok  bitsandbytes==0.49.2
  ok  Pillow==11.3.0
  ok  pycocotools==2.0.11
  ok  scipy==1.16.3
  ok  torchao==0.17.0

  CUDA device  : NVIDIA A100-SXM4-40GB
  VRAM         : 42.4 GB
  DEVICE       : cuda


  ok  Qwen2_5_VLForConditionalGeneration importable
  ok  AutoModelForZeroShotObjectDetection importable (v3.0 grounding)
  ok  sam2 importable (build_sam2, SAM2ImagePredictor)


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


  ok  diffusers + peft components importable
  ok  pycocotools.mask importable
  ok  scipy.ndimage importable (v3.0 mask post-processing)
  ok  CFG in scope

§1.2 passed — all imports and versions OK (v3.0).


## §3.1 — Core Utilities (Phase 3 Subset)
Includes RLE decode, bbox helpers, `QWEN_SPECIAL_START`, `load_json`, `save_json`.

Warning: `QWEN_SPECIAL_START = 151643` must match Phase 1 §11.1 — do not change.

In [3]:
# -- §3.1  Core utilities -- Phase 3 subset ----------------------------------------
# CRITICAL: bbox helpers and QWEN_SPECIAL_START must be byte-identical
# to Phase 1 §3.1 and §11.1. Any change breaks train/inference consistency.

import io, json, logging
import numpy as np
from pathlib import Path
from typing import Any, Optional
from PIL import Image

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger('pipeline')

import pycocotools.mask as mask_utils

# Token IDs < 151643 : regular text tokens -- KEPT for mean pooling
# Token IDs >= 151643: special tokens (<|image_pad|>, <|im_start|>, etc.) -- EXCLUDED
# Confirmed from Phase 1 §11.1. Do NOT change without recomputing all hidden states.
QWEN_SPECIAL_START = 151643

KNOWN_EDIT_TYPES       = set(CFG.expected_edit_types)
GLOBAL_ADJUST_KEYWORDS = set(CFG.global_adjust_keywords)


def decode_rle_mask(rle: dict) -> Optional[np.ndarray]:
    '''Decode COCO-format RLE dict to binary uint8 mask (H x W). Returns None on failure.'''
    if rle is None:
        return None
    try:
        rle_copy = dict(rle)
        if isinstance(rle_copy.get('counts'), str):
            rle_copy['counts'] = rle_copy['counts'].encode('utf-8')
        mask = mask_utils.decode(rle_copy)
        assert mask.ndim == 2, f'Expected 2-D mask, got {mask.shape}'
        return mask
    except Exception as e:
        log.debug(f'RLE decode failed: {e}')
        return None


def bbox_area_absolute(bbox_xyxy: list) -> float:
    x1, y1, x2, y2 = bbox_xyxy
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def clip_bbox_to_image(bbox_xyxy: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = bbox_xyxy
    return [max(0, min(x1, W)), max(0, min(y1, H)),
            max(0, min(x2, W)), max(0, min(y2, H))]


def bbox_to_relative(bbox_xyxy: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = clip_bbox_to_image(bbox_xyxy, W, H)
    return [round(x1/W*1000), round(y1/H*1000), round(x2/W*1000), round(y2/H*1000)]


def bbox_relative_to_absolute(bbox_rel: list, W: int, H: int) -> list:
    '''Convert [0,1000] relative bbox to absolute pixel coords.

    Used in Phase 3 to convert VLM bbox output to SAM2 input.
    SAM2 requires absolute pixel coordinates; VLM outputs [0,1000] relative scale.
    Clamps to image bounds to handle out-of-range VLM predictions.
    '''
    x1, y1, x2, y2 = bbox_rel
    abs_bbox = [round(x1/1000*W), round(y1/1000*H), round(x2/1000*W), round(y2/1000*H)]
    return clip_bbox_to_image(abs_bbox, W, H)


def is_global_edit(edit_type: str, edit_description: str) -> bool:
    '''Return True if this is a global (full-image) edit -- no localised mask needed.'''
    et = (edit_type or '').strip().lower()
    if et in {'style', 'background'}:
        return True
    if et == 'adjust':
        desc_lower = (edit_description or '').lower()
        if any(kw in desc_lower for kw in GLOBAL_ADJUST_KEYWORDS):
            return True
    return False


def is_full_image_bbox(bbox_rel: list) -> bool:
    '''Return True if bbox covers >=95% of the full image in [0,1000] scale.
    Used to detect when VLM predicted a full-image bbox (global edit).
    '''
    x1, y1, x2, y2 = bbox_rel
    return x1 <= 50 and y1 <= 50 and x2 >= 950 and y2 >= 950


def save_json(obj: Any, path: Path, indent: int = 2) -> None:
    '''Atomic JSON write (tmp then rename).'''
    path = Path(path)
    tmp  = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=indent, ensure_ascii=False))
    tmp.rename(path)


def load_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Expected JSON not found: {path}')
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as e:
        raise ValueError(f'Corrupt JSON at {path}: {e}') from e


# -- Smoke test -----------------------------------------------------------
assert QWEN_SPECIAL_START == 151643
assert bbox_relative_to_absolute([500, 250, 750, 875], 1920, 1080) == [960, 270, 1440, 945]
_rel = bbox_to_relative([960, 270, 1440, 945], 1920, 1080)
assert _rel == [500, 250, 750, 875], f'Round-trip bbox failed: {_rel}'
assert is_global_edit('style', 'make it painterly')
assert not is_global_edit('remove', 'remove the car')
assert is_full_image_bbox([0, 0, 1000, 1000])
assert not is_full_image_bbox([100, 100, 800, 800])
print('§3.1 utilities loaded:')
print(f'  QWEN_SPECIAL_START = {QWEN_SPECIAL_START}')
print('  decode_rle_mask, bbox_relative_to_absolute, bbox_to_relative')
print('  is_global_edit, is_full_image_bbox, load_json, save_json')
print('  ok  all smoke tests passed')


§3.1 utilities loaded:
  QWEN_SPECIAL_START = 151643
  decode_rle_mask, bbox_relative_to_absolute, bbox_to_relative
  is_global_edit, is_full_image_bbox, load_json, save_json
  ok  all smoke tests passed


## §8.1 — Phase 1 Prompt Template (Read-Only)
**MUST be byte-identical to Phase 1 §8.1 and Phase 2 §8.1.**
Any change invalidates `PHASE1_TEMPLATE_HASH` and breaks conditioning alignment.

Phase 3 uses `build_messages()` at inference with `annotations=[]` (no ground-truth
regions available at runtime). The model predicts bbox from visual context alone.
Both generation and hidden-state extraction use this same no-annotation prompt.
This is a known train/inference distribution gap — see notebook header.

In [4]:
# -- §8.1  Phase 1 prompt template -- BYTE-IDENTICAL to Phase 1 §8.1 -----------
#
# CRITICAL: Do NOT create a second implementation. Any change here invalidates
# PHASE1_TEMPLATE_HASH and means Phase 3 conditioning diverges from training.
#
# Phase 3 distribution gap (known, acknowledged in spec §7):
#   Training hidden states were extracted with full region lists in the prompt.
#   Inference extracts hidden states WITHOUT region lists (not available at runtime).
#   Both the VLM generation and hidden-state extraction use the same no-annotation
#   prompt -- they are internally consistent with each other.
#   The gap is between the training-time prompt distribution and inference-time.

import hashlib as _hashlib
import json as _json_mod

PHASE1_SYSTEM_PROMPT = (
    'You are an image editing assistant. '
    'Given a source image, a list of segmentation regions with bounding boxes, '
    'and a natural-language edit instruction, '
    'identify the target region and predict the edit operation as a JSON object.\n\n'
    'The segmentation regions are listed as:\n'
    '  - <class_name>: [x1, y1, x2, y2]\n'
    'where coordinates are in [0, 1000] relative scale '
    '(0 = top-left, 1000 = bottom-right).\n\n'
    'JSON schema:\n'
    '{\n'
    '  "edit_type": "<type>",          '
    '// one of: action, add, adjust, background, content,\n'
    '//         hybrid, reference, remove, replace, style, version\n'
    '  "bbox": [x1, y1, x2, y2],       '
    '// the bounding box of the target region in [0, 1000] coordinates\n'
    '//   (select the region from the list above that best matches the instruction)\n'
    '  "edit_description": "<text>"    '
    '// precise description of what to edit\n'
    '}\n\n'
    'Output ONLY valid JSON. No explanation, no markdown code fences.'
)

PHASE1_USER_REGIONS_HEADER = 'Segmentation regions:\n'
PHASE1_USER_PREFIX          = 'Edit instruction: '

_tpl_str = PHASE1_SYSTEM_PROMPT + '|SEP|' + PHASE1_USER_REGIONS_HEADER + '|SEP|' + PHASE1_USER_PREFIX
PHASE1_TEMPLATE_HASH = _hashlib.sha256(_tpl_str.encode()).hexdigest()[:16]
print(f'PHASE1_TEMPLATE_HASH = {PHASE1_TEMPLATE_HASH!r}')
print('  Must match Phase 2 hidden_states_meta.json["phase1_template_hash"] (checked in §20).')


def format_regions_text(annotations: list, seg_W: int, seg_H: int) -> str:
    '''Format segmentation regions as [0,1000] region list.
    Identical to Phase 1 §8.1. Returns empty string when annotations is empty.
    '''
    if not annotations:
        return ''
    lines = [PHASE1_USER_REGIONS_HEADER]
    for ann in annotations:
        class_name = (ann.get('class_name') or 'unknown').strip()
        raw_bbox   = ann.get('bbox') or []
        if len(raw_bbox) != 4:
            continue
        rel_bbox = bbox_to_relative(raw_bbox, seg_W, seg_H)
        lines.append(f'  - {class_name}: {rel_bbox}\n')
    return ''.join(lines)


def build_messages(
    source_image,
    instruction_text: str,
    annotations: list = None,
    seg_dims: tuple   = None,
) -> list:
    '''Build Qwen2.5-VL chat messages. Byte-identical to Phase 1 §8.1.

    Phase 3 calls with annotations=None -- region list omitted gracefully.
    The model predicts bbox from visual understanding + fine-tuned weights.
    '''
    if isinstance(source_image, (str, Path)):
        source_image = Image.open(source_image).convert('RGB')
    elif not isinstance(source_image, Image.Image):
        raise TypeError(f'source_image must be PIL Image or Path, got {type(source_image)}')

    seg_W, seg_H = seg_dims if seg_dims else (1, 1)
    if seg_dims is None and annotations:
        log.warning(
            'build_messages: annotations provided but seg_dims is None. '
            'Region bboxes will be wrong. Always pass seg_dims when annotations present.'
        )

    regions_text = format_regions_text(annotations or [], seg_W, seg_H)
    user_text    = regions_text + PHASE1_USER_PREFIX + instruction_text

    return [
        {'role': 'system', 'content': PHASE1_SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': source_image},
                {'type': 'text',  'text':  user_text},
            ],
        },
    ]


# -- Smoke tests ----------------------------------------------------------
_t = Image.new('RGB', (32, 32), (100, 100, 100))
_m_no_ann = build_messages(_t, 'remove the car')
assert PHASE1_USER_REGIONS_HEADER not in _m_no_ann[1]['content'][1]['text'], (
    'No-annotation path must NOT emit region header'
)
assert PHASE1_USER_PREFIX in _m_no_ann[1]['content'][1]['text']
_m_ann = build_messages(
    _t, 'remove the car',
    annotations=[{'class_name': 'car', 'bbox': [100, 100, 400, 300]}],
    seg_dims=(512, 512),
)
assert PHASE1_USER_REGIONS_HEADER in _m_ann[1]['content'][1]['text']
print('  ok  build_messages no-annotation path (Phase 3 inference path)')
print('  ok  build_messages annotation path')
print(f'\nPhase 3 user turn (no annotations):')
print(_m_no_ann[1]['content'][1]['text'])


PHASE1_TEMPLATE_HASH = 'e8964e0ced3f9891'
  Must match Phase 2 hidden_states_meta.json["phase1_template_hash"] (checked in §20).
  ok  build_messages no-annotation path (Phase 3 inference path)
  ok  build_messages annotation path

Phase 3 user turn (no annotations):
Edit instruction: remove the car


## §20 — Pre-Flight Checks
**Gate:** all checks must pass before §21 (SAM2) or §22 (VLM loading).
Validates Phase 1 checkpoint, Phase 2 checkpoint, and template hash consistency.

In [5]:
# -- §20.1  Pre-flight checks before any model loading -------------------------
#
# v3.0: also notes the grounding-model id. The grounding model is downloaded
# lazily by §22.5 from the HF hub, so absence is not a pre-flight failure.

from pathlib import Path
import json as _json

_pf_failures = []

def _pf_check(condition: bool, msg: str):
    if not condition:
        _pf_failures.append(f'  FAIL  {msg}')
    else:
        print(f'  ok    {msg}')

print('=' * 60)
print('§20.1  Pre-flight checks (v3.0)')
print('=' * 60)

# -- 1. Phase 1 checkpoint exists ------------------------------------------
_p1_adapter = CFG.ckpt_phase1 / 'adapter_config.json'
_p1_model   = CFG.ckpt_phase1 / 'adapter_model.safetensors'
_pf_check(_p1_adapter.exists(),
          f'Phase 1 adapter_config.json: {_p1_adapter}')
_pf_check(_p1_model.exists(),
          f'Phase 1 adapter_model.safetensors: {_p1_model}')

# -- 2. Phase 1 template hash in adapter_config ---------------------------
_p1_hash_ok = False
if _p1_adapter.exists():
    try:
        _adapter_cfg = _json.loads(_p1_adapter.read_text())
        _stored_hash = _adapter_cfg.get('phase1_template_hash')
        if _stored_hash is None:
            _pf_failures.append(
                '  WARN  phase1_template_hash not in adapter_config.json. '
                'Was Phase 1 §10.2 run with the current notebook? '
                'Inference will proceed but template drift is undetected.'
            )
        elif _stored_hash != PHASE1_TEMPLATE_HASH:
            _pf_failures.append(
                f'  FAIL  Template hash mismatch: '
                f'adapter has {_stored_hash!r}, current is {PHASE1_TEMPLATE_HASH!r}. '
                f'Inference prompt does not match training prompt. '
                f'Re-run Phase 1 with the current Phase 3 §8.1 template.'
            )
        else:
            _p1_hash_ok = True
            print(f'  ok    Phase 1 template hash: {_stored_hash!r}')
    except Exception as _e:
        _pf_failures.append(f'  FAIL  Could not read adapter_config.json: {_e}')

# -- 3. Phase 2 final checkpoint exists ------------------------------------
_p2_adapter_pt  = CFG.ckpt_phase2_final / 'vlm_adapter.pt'
_p2_training_st = CFG.ckpt_phase2_final / 'training_state.json'
_p2_unet_hf     = CFG.ckpt_phase2_final / 'unet_lora_hf'
_p2_unet_pt     = CFG.ckpt_phase2_final / 'unet_lora.pt'

_pf_check(_p2_adapter_pt.exists(),
          f'Phase 2 vlm_adapter.pt: {_p2_adapter_pt}')
_pf_check(_p2_training_st.exists(),
          f'Phase 2 training_state.json: {_p2_training_st}')
_p2_unet_exists = _p2_unet_hf.exists() or _p2_unet_pt.exists()
_pf_check(_p2_unet_exists,
          f'Phase 2 UNet LoRA (unet_lora_hf/ or unet_lora.pt): {CFG.ckpt_phase2_final}')

# -- 4. Phase 2 template hash in training_state ----------------------------
if _p2_training_st.exists():
    try:
        _p2_state = _json.loads(_p2_training_st.read_text())
        _p2_hash  = _p2_state.get('template_hash')
        if _p2_hash and _p2_hash != PHASE1_TEMPLATE_HASH:
            _pf_failures.append(
                f'  FAIL  Phase 2 training_state template_hash={_p2_hash!r} '
                f'!= current {PHASE1_TEMPLATE_HASH!r}. '
                'Phase 2 was trained with a different prompt template than Phase 3 uses.'
            )
        else:
            _p2_hash_str = _p2_hash or '(not in checkpoint)'
            print(f'  ok    Phase 2 template hash: {_p2_hash_str!r}')
            _p2_step = _p2_state.get('global_step', '?')
            _p2_loss = _p2_state.get('loss', '?')
            print(f'        trained step={_p2_step}, final_loss={_p2_loss}')
    except Exception as _e:
        _pf_failures.append(f'  FAIL  Could not read training_state.json: {_e}')

# -- 5. SAM2 checkpoint on Drive -------------------------------------------
if CFG.sam2_checkpoint.exists():
    print(f'  ok    SAM2 checkpoint on Drive: {CFG.sam2_checkpoint}')
else:
    _pf_failures.append(
        f'  WARN  SAM2 checkpoint on Drive: {CFG.sam2_checkpoint} '
        f'(missing — run §21.1 to download before §26 SAM2 mask)'
    )

# -- 6. Filtered manifest --------------------------------------------------
_pf_check(CFG.filtered_manifest_path.exists(),
          f'Filtered manifest: {CFG.filtered_manifest_path}')

# -- 7. v3.0: grounding model --------------------------------------------
# We only NOTE the model id; weights download lazily on first use.
print(f'  note  Grounding model (downloads on first §22.5 run): '
      f'{CFG.grounding_model_id}')

# -- Gate ------------------------------------------------------------------
print()
if _pf_failures:
    _warn_only = [f for f in _pf_failures if f.strip().startswith('WARN')]
    _hard_fail = [f for f in _pf_failures if f.strip().startswith('FAIL')]
    if _warn_only:
        for w in _warn_only:
            print(w)
    if _hard_fail:
        raise RuntimeError(
            'Pre-flight checks FAILED:\n' + '\n'.join(_hard_fail) +
            '\n\nFix the above issues before loading any model.'
        )

_PREFLIGHT_PASSED = True
print('ok  All pre-flight checks passed. Proceed to §21 (SAM2), §22 (VLM), §22.5 (Grounding).')


§20.1  Pre-flight checks (v3.0)
  ok    Phase 1 adapter_config.json: /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm/adapter_config.json
  ok    Phase 1 adapter_model.safetensors: /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm/adapter_model.safetensors
  ok    Phase 2 vlm_adapter.pt: /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final/vlm_adapter.pt
  ok    Phase 2 training_state.json: /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final/training_state.json
  ok    Phase 2 UNet LoRA (unet_lora_hf/ or unet_lora.pt): /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final
  ok    Phase 2 template hash: 'e8964e0ced3f9891'
        trained step=2198, final_loss=0.03378298133611679
  ok    SAM2 checkpoint on Drive: /content/drive/MyDrive/img_edit_pipeline/sam2/sam2.1_hiera_large.pt
  ok    Filtered manifest: /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/samp

In [6]:
# -- §20.2  Load filtered manifest into manifest_phase3 --------------------
#
# manifest_phase3 is the authoritative sample list for all Phase 3 inference
# cells (§25.3, §26.2, §28.2, §29.1). It is loaded from samples_filtered.json —
# the same manifest produced by Phase 0 §7.3 and consumed by Phase 2.
#
# Field schema (Phase 0/1/2 canonical names):
#   source_image     — path relative to CFG.data_dir for the source image
#   edit_instruction — natural-language edit instruction passed to the VLM
#   sample_id        — unique string identifier for the sample
#   shard_id         — VLM hidden-state shard name (None for tok-failure samples)
#   row_index        — row index within the shard tensor
#
# Restart safety: JSON load only — safe to rerun at any time.

assert CFG.filtered_manifest_path.exists(), (
    f"Filtered manifest not found: {CFG.filtered_manifest_path}\n"
    "Run Phase 0 §7.3 to build samples_filtered.json before Phase 3."
)

manifest_phase3 = load_json(CFG.filtered_manifest_path)
assert len(manifest_phase3) > 0, (
    "manifest_phase3 is empty — re-run Phase 0 §7.3 to rebuild the manifest."
)

print(f"manifest_phase3 loaded: {len(manifest_phase3):,} samples")
print(f"  Source: {CFG.filtered_manifest_path}")

# Spot-check required field presence so downstream cells fail fast if the
# manifest was generated by an incompatible Phase 0 version.
_m0 = manifest_phase3[0]
for _required_field in ('sample_id', 'source_image', 'edit_instruction'):
    assert _required_field in _m0, (
        f"Expected field '{_required_field}' missing from manifest entry.\n"
        f"Keys present: {list(_m0.keys())}\n"
        "Manifest may have been generated by an incompatible Phase 0 version."
    )

print(f"  Schema check passed (sample_id, source_image, edit_instruction present)")
print(f"  First sample: {_m0['sample_id']!r}")


manifest_phase3 loaded: 8,691 samples
  Source: /content/drive/MyDrive/img_edit_pipeline/data/imgedit_subset/samples_filtered.json
  Schema check passed (sample_id, source_image, edit_instruction present)
  First sample: '0000001'


## §21 — SAM2: Installation & Model Loading
SAM2 is installed from GitHub source (not on PyPI). The checkpoint is cached to
Google Drive so it survives Colab restarts — only the Python package reinstall is
needed after a restart (§21.1), not a re-download.

### §21.1 — SAM2 Install (run once per runtime) + Checkpoint Download

In [7]:
# -- §21.1  SAM2 install from GitHub + checkpoint download to Drive -----------
#
# SAM2 install fragility (known Colab issue):
#   - The sam2 Python package is NOT on PyPI; must be installed from source each runtime.
#   - The checkpoint (1.2 GB) is cached to Drive; subsequent restarts skip download.
#   - If 'from sam2.build_sam import build_sam2' fails, re-run this cell.
#
# Checkpoint URL: SAM2.1 Hiera Large (092824 release).
# The Large model is chosen for accuracy; Small (sam2.1_hiera_small.pt) is faster
# but less precise for complex object boundaries.

import subprocess, sys, os
from pathlib import Path

assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED, (
    '§20 preflight check must pass before §21. Run §20.1 first.'
)

_SAM2_GITHUB_DIR = '/content/sam2'

# -- Install SAM2 from GitHub (every runtime — /content is ephemeral) ------
if not os.path.exists(_SAM2_GITHUB_DIR):
    print('Cloning SAM2 from GitHub...')
    _r = subprocess.run(
        ['git', 'clone', '--quiet',
         'https://github.com/facebookresearch/sam2.git', _SAM2_GITHUB_DIR],
        capture_output=True, text=True
    )
    if _r.returncode != 0:
        raise RuntimeError(f'git clone sam2 failed:\n{_r.stderr[-500:]}')
    print('  ok  git clone sam2')
else:
    print(f'  ok  {_SAM2_GITHUB_DIR} already exists')

_r2 = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', _SAM2_GITHUB_DIR],
    capture_output=True, text=True
)
if _r2.returncode != 0:
    raise RuntimeError(f'pip install sam2 failed:\n{_r2.stderr[-500:]}')
print('  ok  sam2 package installed from source')

# -- Verify import (sys.path guard -- see §1.2 comments) ------------------
# SAM2's build_sam.py raises RuntimeError under TWO independent conditions:
#   1. os.getcwd() equals the parent of the cloned repo (/content in Colab).
#      The check compares os.getcwd() to os.path.dirname(os.path.dirname(__file__))
#      which resolves to /content when the repo lives at /content/sam2/.
#   2. /content (or '' which also resolves to CWD) is in sys.path, allowing
#      'sam2' to resolve to the repo root as a namespace package rather than the
#      installed package at /content/sam2/sam2/.
# BOTH must be fixed. The previous fix only modified sys.path — this is why
# the RuntimeError persisted. The correct fix requires:
#   (a) os.chdir('/root') so os.getcwd() no longer matches /content, AND
#   (b) filter sys.path to remove '' and /content, AND
#   (c) clear any stale sys.modules['sam2'] cache from a prior failed import.
import sys as _sys_sam2, os as _os_sam2

_saved_cwd     = _os_sam2.getcwd()           # /content in Colab
_saved_syspath = _sys_sam2.path[:]

_SAM2_REPO_PARENT = '/content'
_sys_sam2.path = [
    p for p in _sys_sam2.path
    if p != ''                                              # remove '' (= CWD ref)
    and _os_sam2.path.normpath(p) != _os_sam2.path.normpath(_SAM2_REPO_PARENT)
]

# Clear stale sam2 entries a prior failed import may have cached. Without this,
# Python reuses the broken namespace-package entry even after sys.path is fixed.
for _k in [k for k in list(_sys_sam2.modules) if k == 'sam2' or k.startswith('sam2.')]:
    del _sys_sam2.modules[_k]

# Change CWD to /root — no sam2 subdir there, so build_sam.py's CWD check passes.
_os_sam2.chdir('/root')
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
finally:
    _os_sam2.chdir(_saved_cwd)
    _sys_sam2.path = _saved_syspath   # restore regardless of success/failure
try:
    _ = build_sam2
    print('  ok  sam2 importable after install')
except (ImportError, RuntimeError, NameError) as _e:
    raise RuntimeError(
        f'SAM2 import still fails after install: {_e}\n'
        'Try: Runtime -> Restart session, then re-run this cell (§21.1)'
    )

# -- Download checkpoint to Drive (skip if already cached) -----------------
_SAM2_CKPT_URL = (
    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'
)

if CFG.sam2_checkpoint.exists():
    _size_mb = CFG.sam2_checkpoint.stat().st_size / 1e6
    print(f'  ok  SAM2 checkpoint cached on Drive ({_size_mb:.0f} MB): {CFG.sam2_checkpoint}')
else:
    print(f'Downloading SAM2 checkpoint (~1.2 GB) to Drive...')
    print(f'  URL       : {_SAM2_CKPT_URL}')
    print(f'  Saving to : {CFG.sam2_checkpoint}')
    CFG.sam2_dir.mkdir(parents=True, exist_ok=True)
    _r3 = subprocess.run(
        ['wget', '-q', '-O', str(CFG.sam2_checkpoint), _SAM2_CKPT_URL],
        capture_output=True, text=True
    )
    if _r3.returncode != 0:
        CFG.sam2_checkpoint.unlink(missing_ok=True)  # clean partial download
        raise RuntimeError(f'SAM2 checkpoint download failed:\n{_r3.stderr[-300:]}')
    _size_mb = CFG.sam2_checkpoint.stat().st_size / 1e6
    if _size_mb < 100:
        CFG.sam2_checkpoint.unlink(missing_ok=True)
        raise RuntimeError(
            f'Downloaded file is too small ({_size_mb:.1f} MB) -- likely a 404 or partial download.\n'
            f'Check URL: {_SAM2_CKPT_URL}'
        )
    print(f'  ok  SAM2 checkpoint downloaded ({_size_mb:.0f} MB)')

print('\n§21.1 complete. SAM2 ready.')


  ok  /content/sam2 already exists
  ok  sam2 package installed from source
  ok  sam2 importable after install
  ok  SAM2 checkpoint cached on Drive (898 MB): /content/drive/MyDrive/img_edit_pipeline/sam2/sam2.1_hiera_large.pt

§21.1 complete. SAM2 ready.


### §21.2 — Load SAM2 Model & Predictor

In [8]:
# -- §21.2  Load SAM2 model and image predictor --------------------------------
#
# ROOT CAUSE OF MissingConfigException:
#   build_sam2 uses Hydra with pkg:// config resolution. On Python 3.12 with
#   `pip install -e` editable installs, importlib.resources fails to traverse
#   package subdirectories, so Hydra can't find the YAML even if it's on disk.
#
# FIX — use initialize_config_dir (absolute filesystem path):
#   Pre-initialise Hydra with the absolute path to the SAM2 configs directory
#   BEFORE calling build_sam2. build_sam2 skips its own Hydra init when Hydra
#   is already initialised, so compose() resolves configs from the filesystem.
#
# Config directory discovery:
#   The SAM2 repo layout has configs at the REPO ROOT (/content/sam2/configs/)
#   in some releases, and inside the Python package (/content/sam2/sam2/configs/)
#   in others. We probe both directories.
#
# Config FILENAME discovery:
#   The Large config is named "sam2.1_hiera_l.yaml" in the current upstream
#   repo, but past forks/releases used "sam2.1_hiera_large.yaml". We probe
#   both variants and use whichever exists on disk, regardless of which name
#   the user has in CFG. This makes the cell robust to either spelling.

import os, torch
from pathlib import Path

# -- sys.path / CWD guard for SAM2 import ----------------------------------
# build_sam.py raises RuntimeError if os.getcwd() == /content (parent of the
# cloned repo). Fix: chdir to /root for the import only, then restore.
import sys as _sys_sam2, os as _os_sam2

_saved_cwd     = _os_sam2.getcwd()
_saved_syspath = _sys_sam2.path[:]
_SAM2_REPO_PARENT = '/content'

_sys_sam2.path = [
    p for p in _sys_sam2.path
    if p != ''
    and _os_sam2.path.normpath(p) != _os_sam2.path.normpath(_SAM2_REPO_PARENT)
]
for _k in [k for k in list(_sys_sam2.modules) if k == 'sam2' or k.startswith('sam2.')]:
    del _sys_sam2.modules[_k]

_os_sam2.chdir('/root')
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
finally:
    _os_sam2.chdir(_saved_cwd)
    _sys_sam2.path = _saved_syspath

assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED

# -- Locate SAM2 configs directory -----------------------------------------
# The SAM2 repo places configs either at the repo root or inside the package.
# Probe both and use whichever contains the target YAML.
from hydra import initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

_SAM2_GITHUB_DIR   = '/content/sam2'
_SAM2_CONFIG_NAME  = CFG.phase3_sam2_model_cfg   # e.g. "sam2.1/sam2.1_hiera_l"

# Build candidate (config_dir, config_name_without_yaml) pairs.
# The SAM2 repo places configs at one of two locations depending on release:
#   - /content/sam2/configs/             (older repo-root layout)
#   - /content/sam2/sam2/configs/        (current package-internal layout)
# AND the Large config is named "sam2.1_hiera_l.yaml" in the current repo, but
# some past releases / forks used "sam2.1_hiera_large.yaml". To be robust we
# probe BOTH directories AND BOTH filename variants, and resolve to whatever
# actually exists on disk.
_dir_candidates = [
    os.path.join(_SAM2_GITHUB_DIR, 'sam2', 'configs'),  # current layout — try first
    os.path.join(_SAM2_GITHUB_DIR, 'configs'),          # legacy repo-root layout
]

# Filename variants: try the configured name first, then the "_l" <-> "_large"
# alternative. This handles both directions (CFG="_large", repo has "_l", and
# vice versa).
_name_candidates = [_SAM2_CONFIG_NAME]
if _SAM2_CONFIG_NAME.endswith('_large'):
    _name_candidates.append(_SAM2_CONFIG_NAME[:-len('_large')] + '_l')
elif _SAM2_CONFIG_NAME.endswith('_l'):
    _name_candidates.append(_SAM2_CONFIG_NAME[:-len('_l')] + '_large')

_SAM2_CONFIGS_DIR  = None
_SAM2_CONFIG_NAME_RESOLVED = None
_searched = []
for _cand_dir in _dir_candidates:
    for _cand_name in _name_candidates:
        _yaml_path = os.path.join(_cand_dir, _cand_name + '.yaml')
        _searched.append(_yaml_path)
        if Path(_yaml_path).exists():
            _SAM2_CONFIGS_DIR = os.path.abspath(_cand_dir)
            _SAM2_CONFIG_NAME_RESOLVED = _cand_name
            break
    if _SAM2_CONFIGS_DIR is not None:
        break

if _SAM2_CONFIGS_DIR is None:
    _checked = '\n  '.join(_searched)
    raise FileNotFoundError(
        f"SAM2 config YAML for '{_SAM2_CONFIG_NAME}' not found.\n"
        f"Searched:\n  {_checked}\n"
        f"Re-run §21.1 to re-clone the SAM2 repo, then retry §21.2."
    )

if _SAM2_CONFIG_NAME_RESOLVED != _SAM2_CONFIG_NAME:
    print(f"  note  CFG specified '{_SAM2_CONFIG_NAME}' but repo has "
          f"'{_SAM2_CONFIG_NAME_RESOLVED}.yaml' — using the repo filename.")
_SAM2_CONFIG_NAME = _SAM2_CONFIG_NAME_RESOLVED  # use the on-disk name

print(f'Loading SAM2 model...')
print(f'  Config dir : {_SAM2_CONFIGS_DIR}')
print(f'  Config     : {_SAM2_CONFIG_NAME}')
print(f'  Checkpoint : {CFG.sam2_checkpoint}')

# Clear stale Hydra global state (critical for re-runs)
GlobalHydra.instance().clear()

# Pre-initialise Hydra with the absolute filesystem path so build_sam2's
# pkg:// resolver is bypassed entirely.
with initialize_config_dir(config_dir=_SAM2_CONFIGS_DIR, version_base="1.2"):
    sam2_model = build_sam2(
        config_file = _SAM2_CONFIG_NAME,
        ckpt_path   = str(CFG.sam2_checkpoint),
        device      = DEVICE,
    )
sam2_model.eval()
print(f'  ok  SAM2 model loaded ({type(sam2_model).__name__})')

sam2_predictor = SAM2ImagePredictor(sam2_model)
print(f'  ok  SAM2ImagePredictor ready')

# -- SAM2 smoke test -------------------------------------------------------
import numpy as np
_smoke_img_np = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
sam2_predictor.set_image(_smoke_img_np)

_smoke_box = np.array([[0, 0, 64, 64]], dtype=np.float32)
with torch.inference_mode():
    _masks, _scores, _logits = sam2_predictor.predict(
        point_coords     = None,
        point_labels     = None,
        box              = _smoke_box,
        multimask_output = False,
    )

assert _masks.ndim == 3 and _masks.shape[0] == 1, (
    f'SAM2 mask shape unexpected: {_masks.shape}. Expected (1, H, W).'
)
assert _masks.shape[1:] == (64, 64), (
    f'SAM2 mask spatial dims {_masks.shape[1:]} != image dims (64,64)'
)
assert _scores.shape == (1,), f'Scores shape {_scores.shape} != (1,)'
print(f'  ok  SAM2 smoke test: mask shape={tuple(_masks.shape)}, score={_scores[0]:.3f}')
print(f'\n§21.2 SAM2 loaded and verified.')
print(f'  sam2_model:     {type(sam2_model).__name__}')
print(f'  sam2_predictor: {type(sam2_predictor).__name__}')


Loading SAM2 model...
  Config dir : /content/sam2/sam2/configs
  Config     : sam2.1/sam2.1_hiera_l
  Checkpoint : /content/drive/MyDrive/img_edit_pipeline/sam2/sam2.1_hiera_large.pt
  ok  SAM2 model loaded (SAM2Base)
  ok  SAM2ImagePredictor ready
  ok  SAM2 smoke test: mask shape=(1, 64, 64), score=0.995

§21.2 SAM2 loaded and verified.
  sam2_model:     SAM2Base
  sam2_predictor: SAM2ImagePredictor


## §22 — VLM Loading (Phase 1 Fine-Tuned)
Loads Qwen2.5-VL-3B-Instruct with 4-bit NF4 quantization and attaches the
Phase 1 LoRA adapter. The same model is used for both generation (§25) and
hidden-state extraction (§25.2).

In [9]:
# -- §22.1  Load Phase 1 fine-tuned VLM (base + LoRA adapter) -----------------
#
# Loading strategy:
#   1. Load base Qwen2.5-VL-3B-Instruct with 4-bit NF4 quantization
#      (same quantization config as Phase 1 training)
#   2. Attach LoRA adapter from CFG.ckpt_phase1 using PeftModel.from_pretrained
#   3. Set model to eval mode -- inference only, no gradient computation
#
# process_vision_info: extracts PIL images from Qwen message dicts.
# qwen-vl-utils is the primary source; inline fallback if not installed.

import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
)
from peft import PeftModel

assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED

# -- process_vision_info: primary from qwen_vl_utils, fallback inline ------
try:
    from qwen_vl_utils import process_vision_info
    print('  ok  Using qwen_vl_utils.process_vision_info')
except ImportError:
    print('  WARN  qwen_vl_utils not available -- using built-in fallback')

    def process_vision_info(messages: list):
        '''Fallback: extract PIL images from Qwen2.5-VL message content dicts.
        Identical to Phase 1 §9.1 fallback implementation.
        '''
        images = []
        for msg in messages:
            content = msg.get('content', [])
            if not isinstance(content, list):
                continue
            for item in content:
                if not isinstance(item, dict) or item.get('type') != 'image':
                    continue
                img = item.get('image')
                if isinstance(img, Image.Image):
                    images.append(img)
                elif isinstance(img, (str, Path)):
                    images.append(Image.open(img).convert('RGB'))
                else:
                    raise TypeError(
                        f'Unsupported image type {type(img)}. Pass PIL.Image.Image.'
                    )
        return images, []

# -- Quantization config (identical to Phase 1) ----------------------------
_bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

print(f'Loading base model: {CFG.vlm_model_id}')
print('  (First run downloads ~2 GB to HF cache; subsequent runs are instant)')
_base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG.vlm_model_id,
    quantization_config = _bnb_config,
    device_map          = {'': DEVICE},
    torch_dtype         = torch.bfloat16,
    # max_pixels caps image patch count to avoid OOM on high-res images
    # Must match Phase 1 training config (phase1_max_img_pixels)
)
print(f'  ok  base model loaded')

# -- Attach LoRA adapter from Phase 1 checkpoint ---------------------------
print(f'\nLoading Phase 1 LoRA adapter from: {CFG.ckpt_phase1}')
vlm_model = PeftModel.from_pretrained(
    _base_model,
    str(CFG.ckpt_phase1),
    is_trainable = False,   # inference only
)
vlm_model.eval()
print(f'  ok  LoRA adapter attached')

# Verify hidden_size matches CFG.vlm_hidden_dim.
#
# Qwen2.5-VL config layout has changed across transformers versions:
#   * Older releases expose `config.hidden_size` directly on Qwen2_5_VLConfig.
#   * Newer releases (current Colab default) split the config into
#     `config.text_config` and `config.vision_config`; the LM hidden dim
#     lives at `config.text_config.hidden_size` and the top-level attribute
#     is gone, raising AttributeError if accessed naively.
#
# We probe a few well-known locations and fail fast with full diagnostic
# context if none of them resolves -- never silently fall back to CFG.
def _resolve_vlm_hidden_size(mdl) -> int:
    cfg = mdl.config
    candidates = [
        ('config.hidden_size',              lambda c: getattr(c, 'hidden_size', None)),
        ('config.text_config.hidden_size',  lambda c: getattr(getattr(c, 'text_config', None),  'hidden_size', None)),
        ('config.llm_config.hidden_size',   lambda c: getattr(getattr(c, 'llm_config',  None),  'hidden_size', None)),
    ]
    for label, getter in candidates:
        val = getter(cfg)
        if isinstance(val, int) and val > 0:
            print(f'  ok  resolved hidden_size via {label} = {val}')
            return val
    # Fail loudly with diagnostics rather than guessing.
    raise AttributeError(
        f'Could not resolve hidden_size on {type(cfg).__name__}. '
        f'Tried: {[l for l,_ in candidates]}. '
        f'Available top-level attrs (filtered): '
        f'{[a for a in dir(cfg) if not a.startswith("_") and "config" in a.lower() or "hidden" in a.lower()][:20]}'
    )

_actual_hidden = _resolve_vlm_hidden_size(vlm_model)
assert _actual_hidden == CFG.vlm_hidden_dim, (
    f'Model hidden_size={_actual_hidden} != CFG.vlm_hidden_dim={CFG.vlm_hidden_dim}. '
    f'Update CFG.vlm_hidden_dim to {_actual_hidden}.'
)
print(f'  ok  hidden_size={_actual_hidden} matches CFG.vlm_hidden_dim')

# -- Load processor (tokenizer + image processor) -------------------------
# Load from checkpoint dir first (processor was saved with adapter in Phase 1 §10.2).
# Fall back to hub if not present (older Phase 1 checkpoints).
_proc_path = CFG.ckpt_phase1
if not (_proc_path / 'tokenizer_config.json').exists():
    _proc_path = CFG.vlm_model_id
    print(f'  WARN  No processor in checkpoint -- loading from hub: {CFG.vlm_model_id}')

vlm_processor = AutoProcessor.from_pretrained(str(_proc_path))
print(f'  ok  processor loaded from {_proc_path}')

# Freeze all VLM weights (LoRA included) -- no gradient needed at inference
for _p in vlm_model.parameters():
    _p.requires_grad_(False)

print(f'\n§22.1  VLM ready for inference.')
print(f'  Model: {type(vlm_model).__name__}')
print(f'  Quantization: 4-bit NF4 (bfloat16 compute)')
print(f'  Device: {next(vlm_model.parameters()).device}')


  ok  Using qwen_vl_utils.process_vision_info
Loading base model: Qwen/Qwen2.5-VL-3B-Instruct
  (First run downloads ~2 GB to HF cache; subsequent runs are instant)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

  ok  base model loaded

Loading Phase 1 LoRA adapter from: /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm
  ok  LoRA adapter attached
  ok  resolved hidden_size via config.text_config.hidden_size = 2048
  ok  hidden_size=2048 matches CFG.vlm_hidden_dim


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


  ok  processor loaded from /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase1_vlm

§22.1  VLM ready for inference.
  Model: PeftModelForCausalLM
  Quantization: 4-bit NF4 (bfloat16 compute)
  Device: cuda:0


## §22.5 — GroundingDINO Loader (v3.0)

In [12]:
# -- §22.5  Load GroundingDINO (v3.0) ----------------------------------------
#
# Loads `IDEA-Research/grounding-dino-tiny` from the HF hub via
# AutoModelForZeroShotObjectDetection + AutoProcessor. We deliberately use the
# Transformers port instead of the standalone `groundingdino-py` package:
#   * No CUDA-extension build — works on Colab GPUs that lack matching nvcc.
#   * Uses the same AutoProcessor pattern as the rest of the notebook.
#   * Tiny variant is ~170 MB and runs in < 200 ms per image on an A100.
#
# We freeze all weights and load in fp16 to share VRAM with VLM + SAM2 + SD.
# The `run_grounding_dino` helper below is the only call site.

import torch
import numpy as np
from PIL import Image
from typing import List, Tuple
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

assert "_PREFLIGHT_PASSED" in dir() and _PREFLIGHT_PASSED

print(f"Loading GroundingDINO: {CFG.grounding_model_id}")
print("  (first run downloads ~170 MB to HF cache)")

# Processor handles both image preprocessing and tokenizing the text prompt.
grounding_processor = AutoProcessor.from_pretrained(CFG.grounding_model_id)

# Load in fp16 directly on the CUDA device. GroundingDINO ships a single-call
# API; we don't need autocast acrobatics.
#grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
#    CFG.grounding_model_id,
#    torch_dtype=torch.float16,
#).to(DEVICE).eval()

# NOTE: We deliberately load GroundingDINO in fp32, not fp16.
# The HF transformers port has a known dtype-mixing bug in the text branch:
# `text_enhancer_layer.with_pos_embed` adds fp32 sinusoidal position
# embeddings to fp16 hidden states, producing fp32 queries that then hit
# fp16 Linear weights -> "mat1 and mat2 must have the same dtype" RuntimeError.
# The tiny variant is ~700 MB in fp32, which is acceptable alongside the
# rest of the pipeline. Do not "optimize" this back to fp16 without first
# confirming the upstream bug is fixed.
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    CFG.grounding_model_id,
    torch_dtype=torch.float32,
).to(DEVICE).eval()

for _p in grounding_model.parameters():
    _p.requires_grad_(False)

print(f"  ok  GroundingDINO loaded ({type(grounding_model).__name__})")
print(f"      device={next(grounding_model.parameters()).device}, "
      f"dtype={next(grounding_model.parameters()).dtype}")


def _format_grounding_prompt(phrase: str) -> str:
    """GroundingDINO expects a period-terminated, lowercased prompt.

    The HF docs explicitly require this format — without the trailing period
    the model's text-token alignment regresses sharply. We also strip leading
    articles so the model sees the noun head ("the motorcycle" -> "motorcycle.").
    Multi-noun queries can be passed as "motorcycle. text. cake.".
    """
    p = (phrase or "").strip().lower()
    if not p:
        return ""
    # Strip a single leading article — pre-trained DINO is more robust to bare nouns.
    for _art in ("the ", "a ", "an "):
        if p.startswith(_art):
            p = p[len(_art):]
            break
    if not p.endswith("."):
        p = p + "."
    return p


def run_grounding_dino(
    image: Image.Image,
    phrase: str,
) -> List[Tuple[List[float], float]]:
    """Run GroundingDINO and return [(bbox_xyxy_abs, score), ...] sorted by score.

    Args:
        image  : PIL.Image (RGB).
        phrase : free-form text describing the target object(s).

    Returns:
        List of (bbox_xyxy_abs_pixels, score) tuples, max len = grounding_max_boxes.
        Empty list if no detections cleared the threshold.
    """
    prompt = _format_grounding_prompt(phrase)
    if not prompt:
        log.info("run_grounding_dino: empty prompt — skipping detection")
        return []

    img_rgb = image.convert("RGB")
    W, H    = img_rgb.size

    inputs = grounding_processor(
        images=img_rgb,
        text=prompt,
        return_tensors="pt",
    ).to(DEVICE)

    # GroundingDINO expects fp16 image features + fp32 attention masks; the
    # processor outputs fp32 everywhere, so we cast pixel_values only.
    #if "pixel_values" in inputs:
    #    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    # Model is loaded in fp32 (see §22.5 note on the text-branch dtype bug);
    # processor already returns fp32 tensors, so no explicit casting needed.
    # We keep this assertion to fail fast if anyone re-introduces fp16 loading
    # without addressing the text-branch mismatch.
    assert next(grounding_model.parameters()).dtype == torch.float32, (
        "GroundingDINO must be loaded in fp32; see §22.5 comment for why."
    )

    with torch.no_grad():
        outputs = grounding_model(**inputs)

    # post_process_grounded_object_detection returns boxes in xyxy abs coords
    # for the ORIGINAL image size when target_sizes is provided.
    target_sizes = torch.tensor([[H, W]], device=DEVICE)
    #results = grounding_processor.post_process_grounded_object_detection(
    #    outputs,
    #    inputs["input_ids"],
    #    box_threshold=CFG.grounding_box_threshold,
    #    text_threshold=CFG.grounding_text_threshold,
    #    target_sizes=target_sizes,
   # )[0]
   # NOTE: transformers renamed `box_threshold` -> `threshold` in recent
    # versions of GroundingDinoProcessor.post_process_grounded_object_detection.
    # `text_threshold` kept its name. We pass via the new name; if you pin
    # an older transformers, swap back to `box_threshold=`.
    results = grounding_processor.post_process_grounded_object_detection(
        outputs,
        inputs["input_ids"],
        threshold=CFG.grounding_box_threshold,
        text_threshold=CFG.grounding_text_threshold,
        target_sizes=target_sizes,
    )[0]

    boxes  = results["boxes"].detach().cpu().numpy().tolist()   # list of [x1,y1,x2,y2]
    scores = results["scores"].detach().cpu().numpy().tolist()  # list of floats

    if not boxes:
        log.info(
            f"run_grounding_dino: 0 detections for prompt {prompt!r} "
            f"(box_thr={CFG.grounding_box_threshold}, "
            f"text_thr={CFG.grounding_text_threshold})"
        )
        return []

    # Pair, sort by descending score, cap to grounding_max_boxes.
    paired = sorted(zip(boxes, scores), key=lambda bs: -bs[1])
    paired = paired[: CFG.grounding_max_boxes]

    log.info(
        f"run_grounding_dino: prompt={prompt!r} -> {len(paired)} boxes "
        f"(top score={paired[0][1]:.3f})"
    )
    return paired


# -- Smoke test ------------------------------------------------------------
# We test on a 256x256 noise image. With nothing recognizable, we expect the
# model to return zero detections — which is the correct behaviour and proves
# the post-processing call is wired up.
_smoke_img   = Image.fromarray(
    np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8), mode="RGB"
)
_smoke_boxes = run_grounding_dino(_smoke_img, "a cat")
print(f"  ok  GroundingDINO smoke test on noise image: "
      f"{len(_smoke_boxes)} detections (expected 0 for noise)")
print(f"\n§22.5 GroundingDINO ready for inference.")


Loading GroundingDINO: IDEA-Research/grounding-dino-tiny
  (first run downloads ~170 MB to HF cache)


Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

  ok  GroundingDINO loaded (GroundingDinoForObjectDetection)
      device=cuda:0, dtype=torch.float32
  ok  GroundingDINO smoke test on noise image: 1 detections (expected 0 for noise)

§22.5 GroundingDINO ready for inference.


/tmp/ipykernel_3220/36222047.py:168: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  _smoke_img   = Image.fromarray(


## §23 — SD Pipeline Components (Phase 2 Checkpoint)
Loads VAE, UNet (9-channel), CLIP encoder, tokenizer, and DDIM scheduler from
the SD 1.5 inpainting model. Then loads the trained VLMProjectionAdapter and
attaches the UNet LoRA from the Phase 2 final checkpoint.

### §23.1 — Load SD Inpainting Components

In [13]:
# -- §23.1  Load SD inpainting components (VAE, UNet, CLIP, DDIMScheduler) ----
# (v1.9: added fail-fast scheduler parity assertions against Phase 2's effective
#        config. Architecture itself is unchanged from v1.8 — same repo, same
#        9-channel inpainting UNet, same CLIP/VAE.)
#
# Same component loading as Phase 2 §14.1 — identical arguments.
# DDIM replaces DDPM for inference (fewer steps: 20 vs 1000).
#
# CRITICAL: load UNet as fp32 first, then cast LoRA + UNet to fp16 after
# PEFT attach. PeftModel wraps the UNet and is not a UNet2DConditionModel
# instance, so from_pretrained(..., torch_dtype=fp16) skips the cast.
# Explicit cast to fp16 is done in §23.2 after PEFT attach.

import torch
from diffusers import (
    AutoencoderKL, UNet2DConditionModel, DDIMScheduler,
    StableDiffusionInpaintPipeline,
)
from transformers import CLIPTextModel, CLIPTokenizer

assert '_PREFLIGHT_PASSED' in dir() and _PREFLIGHT_PASSED

print(f'Loading SD inpainting components from: {CFG.sd_model_id}')

# VAE -- frozen, fp32 (stability for encode/decode)
vae = AutoencoderKL.from_pretrained(
    CFG.sd_model_id, subfolder='vae', torch_dtype=torch.float32
)
vae.eval()
vae.requires_grad_(False)
vae = vae.to(DEVICE)
print(f'  ok  VAE loaded')

# UNet -- 9-channel inpainting variant (channels: noisy_tgt | mask | masked_src)
# Load fp32; will be cast to fp16 after LoRA attach in §23.2
unet = UNet2DConditionModel.from_pretrained(
    CFG.sd_model_id, subfolder='unet', torch_dtype=torch.float32
)
assert unet.config.in_channels == 9, (
    f'Expected UNet in_channels=9 (inpainting), got {unet.config.in_channels}.\n'
    f'Use runwayml/stable-diffusion-inpainting, not base SD 1.5.'
)
assert unet.config.cross_attention_dim == CFG.sd_cross_attn_dim, (
    f'UNet cross_attention_dim={unet.config.cross_attention_dim} '
    f'!= CFG.sd_cross_attn_dim={CFG.sd_cross_attn_dim}'
)
unet = unet.to(DEVICE)
print(f'  ok  UNet loaded: in_channels={unet.config.in_channels}, '
      f'cross_attention_dim={unet.config.cross_attention_dim}')

# CLIP text encoder + tokenizer -- frozen
sd_tokenizer = CLIPTokenizer.from_pretrained(CFG.sd_model_id, subfolder='tokenizer')
clip = CLIPTextModel.from_pretrained(
    CFG.sd_model_id, subfolder='text_encoder', torch_dtype=torch.float32
)
clip.eval()
clip.requires_grad_(False)
clip = clip.to(DEVICE)
print(f'  ok  CLIP loaded: hidden_size={clip.config.hidden_size}')

# DDIM scheduler -- faster inference (20 steps vs DDPM 1000)
#
# Train/inference parity note (verified against phase2_sd_bridge_training_v1_5):
#   Phase 2 constructs the training scheduler with
#       DDPMScheduler.from_pretrained(CFG.sd_model_id, subfolder='scheduler')
#   and only overrides num_train_timesteps. Despite CFG.phase2_beta_schedule='linear'
#   being defined, that constant is NEVER applied to the scheduler in Phase 2,
#   so the UNet was effectively trained against the runwayml inpainting repo's
#   shipped beta_schedule (= 'scaled_linear'). To preserve train/inference parity
#   we MUST load Phase 3's DDIM scheduler from the same repo+subfolder and assert
#   the betas match what training saw. The assertions below fail-fast on any drift
#   (e.g. someone editing the scheduler config, or the upstream repo changing).
noise_scheduler = DDIMScheduler.from_pretrained(CFG.sd_model_id, subfolder='scheduler')

# --- Fail-fast parity checks against Phase 2 effective config -------------------
# Phase 2 inherited these values from the runwayml/stable-diffusion-inpainting
# scheduler config; if any of them changes here, denoising at inference will
# operate on a different noise schedule than the UNet was trained against.
_EXPECTED_BETA_SCHEDULE = 'scaled_linear'   # inherited by Phase 2 from the repo
_EXPECTED_BETA_START   = 0.00085            # SD 1.5 family default
_EXPECTED_BETA_END     = 0.012              # SD 1.5 family default
_EXPECTED_NUM_TRAIN_TIMESTEPS = 1000        # Phase 2 explicitly sets this

_bs  = noise_scheduler.config.beta_schedule
_b0  = float(noise_scheduler.config.beta_start)
_b1  = float(noise_scheduler.config.beta_end)
_nts = int(noise_scheduler.config.num_train_timesteps)

assert _bs == _EXPECTED_BETA_SCHEDULE, (
    f'beta_schedule mismatch vs Phase 2: got {_bs!r}, expected {_EXPECTED_BETA_SCHEDULE!r}. '
    f'Training UNet against one schedule and sampling with another produces degraded outputs.'
)
assert abs(_b0 - _EXPECTED_BETA_START) < 1e-9, (
    f'beta_start mismatch vs Phase 2: got {_b0}, expected {_EXPECTED_BETA_START}'
)
assert abs(_b1 - _EXPECTED_BETA_END) < 1e-9, (
    f'beta_end mismatch vs Phase 2: got {_b1}, expected {_EXPECTED_BETA_END}'
)
assert _nts == _EXPECTED_NUM_TRAIN_TIMESTEPS, (
    f'num_train_timesteps mismatch vs Phase 2: got {_nts}, expected {_EXPECTED_NUM_TRAIN_TIMESTEPS}'
)

# Inference-only step count; does NOT change the underlying noise schedule.
noise_scheduler.set_timesteps(CFG.phase3_num_inference_steps)

print(f'  ok  DDIMScheduler: {CFG.phase3_num_inference_steps} inference steps, '
      f'beta_schedule={_bs}, beta_start={_b0}, beta_end={_b1}, '
      f'num_train_timesteps={_nts}  (parity vs Phase 2 verified)')

print(f'\n§23.1 SD components loaded. Proceed to §23.2 to load trained adapters.')


Loading SD inpainting components from: runwayml/stable-diffusion-inpainting


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


vae/diffusion_pytorch_model.bin:   0%|          | 0.00/335M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-inpainting
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ok  CLIP loaded: hidden_size=768


scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

  ok  DDIMScheduler: 20 inference steps, beta_schedule=scaled_linear, beta_start=0.00085, beta_end=0.012, num_train_timesteps=1000  (parity vs Phase 2 verified)

§23.1 SD components loaded. Proceed to §23.2 to load trained adapters.


### §23.2 — VLMProjectionAdapter + UNet LoRA (Phase 2 Checkpoint)

In [14]:
# -- §23.2  Load VLMProjectionAdapter and UNet LoRA from Phase 2 checkpoint ---
#
# VLMProjectionAdapter architecture: byte-identical to Phase 2 §14.3.
# Any change here means the loaded weights will mismatch the checkpoint layout.
#
# UNet LoRA loading:
#   Preferred: PeftModel.from_pretrained from unet_lora_hf/ (PEFT format)
#   Fallback:  load_state_dict from unet_lora.pt (manual tensor save)
#              (used when unet.save_pretrained failed in Phase 2 §18)

import torch
import torch.nn as nn
from peft import PeftModel

# -- VLMProjectionAdapter definition (byte-identical to Phase 2 §14.3) -----
class VLMProjectionAdapter(nn.Module):
    '''Projects VLM mean-pooled hidden state to SD cross-attention embedding.

    Input:  (batch, vlm_dim=2048)     float32
    Output: (batch, 1, sd_dim=768)    float32

    Concatenated with CLIP text embeddings (batch,77,768):
        combined = cat([clip_embeds, vlm_proj], dim=1)  -> (batch, 78, 768)
    '''

    def __init__(self, vlm_dim: int = None, sd_dim: int = None):
        super().__init__()
        vlm_dim = vlm_dim or CFG.vlm_hidden_dim        # 2048
        sd_dim  = sd_dim  or CFG.sd_cross_attn_dim     # 768
        intermediate_dim = sd_dim * 2                   # 1536

        self.proj = nn.Sequential(
            nn.Linear(vlm_dim, intermediate_dim, bias=True),
            nn.LayerNorm(intermediate_dim),
            nn.GELU(),
            nn.Linear(intermediate_dim, sd_dim, bias=True),
        )
        # Weight init: small normal (matches Phase 2 definition exactly)
        for layer in self.proj:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, std=0.02)
                nn.init.zeros_(layer.bias)

    def forward(self, vlm_hidden: torch.Tensor) -> torch.Tensor:
        assert vlm_hidden.ndim == 2, (
            f'VLMProjectionAdapter expects (batch, vlm_dim), got {vlm_hidden.shape}'
        )
        return self.proj(vlm_hidden).unsqueeze(1)   # (batch, 1, sd_dim)


# Instantiate and load Phase 2 weights
vlm_adapter = VLMProjectionAdapter().to(DEVICE)
vlm_adapter.load_state_dict(
    torch.load(CFG.ckpt_phase2_final / 'vlm_adapter.pt', map_location=DEVICE)
)
vlm_adapter.eval()
for _p in vlm_adapter.parameters():
    _p.requires_grad_(False)
print(f'  ok  VLMProjectionAdapter loaded from Phase 2 checkpoint')

# Shape check
_test_hs  = torch.randn(2, CFG.vlm_hidden_dim, device=DEVICE)
_test_out = vlm_adapter(_test_hs)
assert _test_out.shape == (2, 1, CFG.sd_cross_attn_dim), (
    f'VLMProjectionAdapter output {_test_out.shape} != (2, 1, {CFG.sd_cross_attn_dim})'
)
print(f'  ok  VLMProjectionAdapter shape: (2,{CFG.vlm_hidden_dim}) -> {tuple(_test_out.shape)}')

# -- UNet LoRA from Phase 2 checkpoint ------------------------------------
_p2_unet_hf = CFG.ckpt_phase2_final / 'unet_lora_hf'
_p2_unet_pt = CFG.ckpt_phase2_final / 'unet_lora.pt'

if _p2_unet_hf.exists():
    # PEFT format (preferred): wraps the UNet with PEFT LoRA
    unet = PeftModel.from_pretrained(unet, str(_p2_unet_hf), is_trainable=False)
    print(f'  ok  UNet LoRA loaded from PEFT format: {_p2_unet_hf}')
elif _p2_unet_pt.exists():
    # Manual fallback: load LoRA state dict into base UNet
    _lora_state = torch.load(_p2_unet_pt, map_location=DEVICE)
    _missing, _unexpected = unet.load_state_dict(_lora_state, strict=False)
    if _unexpected:
        log.warning(f'UNet LoRA: {len(_unexpected)} unexpected keys in state dict')
    print(f'  ok  UNet LoRA loaded from manual .pt: {_p2_unet_pt}')
    print(f'      LoRA keys loaded: {len(_lora_state)}')
else:
    raise FileNotFoundError(
        f'No UNet LoRA checkpoint found at:\n'
        f'  {_p2_unet_hf} (PEFT format)\n'
        f'  {_p2_unet_pt} (manual .pt)\n'
        'Re-run Phase 2 §18.2 to complete training.'
    )

unet.eval()
for _p in unet.parameters():
    _p.requires_grad_(False)

# -- Cast UNet to fp16 (PeftModel is not auto-cast by from_pretrained) -----
# This is the same fix as Phase 2 §19: PeftModel wraps UNet2DConditionModel,
# so from_pretrained dtype= does not apply. Cast explicitly here.
unet = unet.to(dtype=torch.float16)
print(f'  ok  UNet cast to fp16 (dtype={next(unet.parameters()).dtype})')

print(f'\n§23.2  Phase 2 adapters loaded and verified.')
print(f'  VLMProjectionAdapter: frozen, DEVICE={DEVICE}')
print(f'  UNet LoRA:            frozen, dtype=fp16')


  ok  VLMProjectionAdapter loaded from Phase 2 checkpoint
  ok  VLMProjectionAdapter shape: (2,2048) -> (2, 1, 768)
  ok  UNet LoRA loaded from PEFT format: /content/drive/MyDrive/img_edit_pipeline/checkpoints/phase2_diffusion_v2_r16/final/unet_lora_hf
  ok  UNet cast to fp16 (dtype=torch.float16)

§23.2  Phase 2 adapters loaded and verified.
  VLMProjectionAdapter: frozen, DEVICE=cuda
  UNet LoRA:            frozen, dtype=fp16


### §23.3 — build_combined_conditioning (byte-identical to Phase 2 §16)

In [15]:
# -- §23.3  Conditioning pipeline -- CLIP + VLM projection ------------------
#
# SPEC REQUIREMENT: 'Protect against train-inference mismatch.'
# This function MUST be byte-identical to Phase 2 §16.
# Using do_cfg=True at inference and do_cfg=False during training.
#
# Inference CFG pattern (called from run_inpainting §27.1):
#   cond = build_combined_conditioning([instr], vlm_hs, device, do_cfg=True)
#   # -> (2, 78, 768): rows 0 = uncond, rows 1 = cond
#   noise_pred_2b = unet(unet_input_tiled, t, encoder_hidden_states=cond).sample
#   # -> (2, 4, H/8, W/8)
#   noise_pred_uncond, noise_pred_cond = noise_pred_2b.chunk(2)
#   noise_pred = noise_pred_uncond + scale * (noise_pred_cond - noise_pred_uncond)

import torch


def build_combined_conditioning(
    instructions: list,
    vlm_hiddens: torch.Tensor,
    device: str,
    do_cfg: bool = False,
) -> torch.Tensor:
    '''Build combined (CLIP + VLM) conditioning tensor. Byte-identical to Phase 2 §16.

    Args:
        instructions : List[str] of length batch
        vlm_hiddens  : (batch, vlm_hidden_dim) float32
        device       : 'cuda' or 'cpu'
        do_cfg       : If True, prepend unconditional branch (inference only).
                       MUST be False during training.

    Returns:
        Training   (do_cfg=False): (batch, 78, 768)
        Inference  (do_cfg=True) : (2*batch, 78, 768)
                                   rows 0..B-1 = unconditional
                                   rows B..2B-1 = conditional
    '''
    batch = len(instructions)
    assert vlm_hiddens.shape == (batch, CFG.vlm_hidden_dim), (
        f'vlm_hiddens shape {vlm_hiddens.shape} != ({batch}, {CFG.vlm_hidden_dim})'
    )

    # CLIP text encoding
    clip_tokens = sd_tokenizer(
        instructions,
        padding        = 'max_length',
        truncation     = True,
        max_length     = 77,
        return_tensors = 'pt',
    ).to(device)
    with torch.no_grad():
        clip_embeds = clip(**clip_tokens).last_hidden_state  # (batch, 77, 768)

    # VLM projection
    vlm_proj = vlm_adapter(vlm_hiddens.to(device).float())   # (batch, 1, 768)

    # Conditional branch
    cond = torch.cat([clip_embeds, vlm_proj], dim=1)          # (batch, 78, 768)

    if not do_cfg:
        return cond

    # Unconditional branch (inference CFG only)
    uncond_tokens = sd_tokenizer(
        [''] * batch,
        padding        = 'max_length',
        truncation     = True,
        max_length     = 77,
        return_tensors = 'pt',
    ).to(device)
    with torch.no_grad():
        uncond_clip = uncond_tokens
        uncond_clip = clip(**uncond_tokens).last_hidden_state  # (batch, 77, 768)

    # Null VLM token: zeros -> no VLM-specific signal in unconditional branch
    null_vlm = torch.zeros(batch, 1, CFG.sd_cross_attn_dim, device=device)
    uncond   = torch.cat([uncond_clip, null_vlm], dim=1)       # (batch, 78, 768)

    # Stack: [uncond, cond] along batch dimension
    return torch.cat([uncond, cond], dim=0)                    # (2*batch, 78, 768)


# -- Smoke test -----------------------------------------------------------
_test_instr  = ['change the shirt to blue', 'remove the car']
_test_vlm_hs = torch.randn(2, CFG.vlm_hidden_dim, device=DEVICE)

_cond_train  = build_combined_conditioning(_test_instr, _test_vlm_hs, DEVICE, do_cfg=False)
assert _cond_train.shape == (2, 78, CFG.sd_cross_attn_dim), (
    f'Training cond {_cond_train.shape} != (2,78,{CFG.sd_cross_attn_dim})'
)
print(f'  ok  training cond:   {tuple(_cond_train.shape)}')

_cond_infer  = build_combined_conditioning(_test_instr, _test_vlm_hs, DEVICE, do_cfg=True)
assert _cond_infer.shape == (4, 78, CFG.sd_cross_attn_dim), (
    f'Inference cond {_cond_infer.shape} != (4,78,{CFG.sd_cross_attn_dim})'
)
_uncond_half = _cond_infer[:2]
_cond_half   = _cond_infer[2:]
# Unconditional VLM token (last token in seq) should be zeros
assert _uncond_half[:, -1, :].abs().max().item() < 1e-6, (
    'Unconditional VLM token is not zero -- check null_vlm in build_combined_conditioning'
)
print(f'  ok  inference cond:  {tuple(_cond_infer.shape)} (CFG: uncond||cond)')
print(f'  ok  uncond VLM token zeros: {_uncond_half[:,-1,:].abs().max().item():.2e}')
print(f'\nbuild_combined_conditioning verified (byte-identical to Phase 2 §16).')


  ok  training cond:   (2, 78, 768)
  ok  inference cond:  (4, 78, 768) (CFG: uncond||cond)
  ok  uncond VLM token zeros: 0.00e+00

build_combined_conditioning verified (byte-identical to Phase 2 §16).


## §24 — Architecture Smoke Test
Full forward chain: random VLM hidden state + instruction + mock image/mask
through VLMProjectionAdapter + CLIP + UNet → noise prediction. All shapes asserted.
Run before §25 to confirm all components work together.

In [16]:
# -- §24.1  Architecture smoke test (all Phase 3 components together) ---------
#
# SPEC REQUIREMENT: 'Before writing any training loop, first ensure that the
# notebook can run one forward pass through the diffusion UNet with correct shapes.'
# Phase 3 equivalent: full forward chain before any inference function is called.

import torch, torch.nn.functional as F
import numpy as np

_B   = 1
_RES = CFG.phase3_resolution   # 512
_LAT = _RES // 8               # 64

print('=' * 60)
print('§24.1  Architecture smoke test (Phase 3)')
print('=' * 60)

# [1] VLM hidden state (simulates extract_hidden_state_inference output)
_vlm_hs = torch.randn(_B, CFG.vlm_hidden_dim).to(DEVICE)
print(f'  [1] VLM hidden state    : {tuple(_vlm_hs.shape)}')

# [2] VLM projection
with torch.no_grad():
    _vlm_proj = vlm_adapter(_vlm_hs)    # (1, 1, 768)
assert _vlm_proj.shape == (_B, 1, CFG.sd_cross_attn_dim)
print(f'  [2] VLM projected       : {tuple(_vlm_proj.shape)}')

# [3] CLIP text encoding
_instrs = ['change the shirt to blue']
_clip_tokens = sd_tokenizer(_instrs, padding='max_length', truncation=True,
                            max_length=77, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    _clip_embeds = clip(**_clip_tokens).last_hidden_state  # (1, 77, 768)
assert _clip_embeds.shape == (_B, 77, CFG.sd_cross_attn_dim)
print(f'  [3] CLIP text embeds    : {tuple(_clip_embeds.shape)}')

# [4] Combined conditioning (CFG=True for inference)
_combined_cond = build_combined_conditioning(_instrs, _vlm_hs.cpu(), DEVICE, do_cfg=True)
assert _combined_cond.shape == (2*_B, 78, CFG.sd_cross_attn_dim), (
    f'Combined cond shape {_combined_cond.shape} != ({2*_B}, 78, {CFG.sd_cross_attn_dim})'
)
print(f'  [4] Combined cond (CFG) : {tuple(_combined_cond.shape)} [uncond||cond]')

# [5] Mock source image + target VAE encode (fp32 for stability)
_mock_src = torch.rand(_B, 3, _RES, _RES, device=DEVICE) * 2 - 1
with torch.no_grad():
    _src_latents = vae.encode(_mock_src.float()).latent_dist.sample() * CFG.vae_scale_factor
assert _src_latents.shape == (_B, 4, _LAT, _LAT)
print(f'  [5] Source latents      : {tuple(_src_latents.shape)}')

# [6] Mask at image res -> latent res
_mock_mask = torch.zeros(_B, 1, _RES, _RES, device=DEVICE)
_mock_mask[:, :, 100:400, 150:350] = 1.0
_masked_src    = _mock_src * (1.0 - _mock_mask)
with torch.no_grad():
    _masked_latents = vae.encode(_masked_src.float()).latent_dist.sample() * CFG.vae_scale_factor
_mask_latent = F.interpolate(_mock_mask, size=(_LAT, _LAT), mode='nearest')
assert _mask_latent.shape == (_B, 1, _LAT, _LAT)
print(f'  [6] Mask (latent size)  : {tuple(_mask_latent.shape)}')

# [7] Initial noise latent (what DDIM denoising starts from)
_latents_init = torch.randn(_B, 4, _LAT, _LAT, device=DEVICE, dtype=torch.float16)
# Scale by scheduler's initial noise sigma
_latents_init = _latents_init * noise_scheduler.init_noise_sigma

# Tile for CFG: duplicate latents, mask, masked_src for batch=2
_t_test = noise_scheduler.timesteps[0]
_latents_2x    = torch.cat([_latents_init] * 2)        # (2, 4, 64, 64)
_mask_2x       = torch.cat([_mask_latent.half()] * 2)  # (2, 1, 64, 64)
_masked_2x     = torch.cat([_masked_latents.half()] * 2)  # (2, 4, 64, 64)
_unet_input_2x = torch.cat([_latents_2x, _mask_2x, _masked_2x], dim=1)  # (2, 9, 64, 64)

assert _unet_input_2x.shape == (2*_B, 9, _LAT, _LAT), (
    f'UNet input shape {_unet_input_2x.shape} != ({2*_B}, 9, {_LAT}, {_LAT})'
)
print(f'  [7] UNet 9-ch input (x2): {tuple(_unet_input_2x.shape)} (noisy|mask|masked_src)')

# [8] UNet forward pass
with torch.no_grad():
    _noise_pred = unet(
        _unet_input_2x.half(),
        _t_test,
        encoder_hidden_states=_combined_cond.half(),
    ).sample
assert _noise_pred.shape == (2*_B, 4, _LAT, _LAT), (
    f'Noise pred shape {_noise_pred.shape} != ({2*_B}, 4, {_LAT}, {_LAT})'
)
print(f'  [8] Noise prediction    : {tuple(_noise_pred.shape)}')

# [9] CFG combination
_pred_uncond, _pred_cond = _noise_pred.chunk(2)
_pred_guided = _pred_uncond + CFG.phase3_guidance_scale * (_pred_cond - _pred_uncond)
assert _pred_guided.shape == (_B, 4, _LAT, _LAT)
print(f'  [9] CFG guided pred     : {tuple(_pred_guided.shape)} (scale={CFG.phase3_guidance_scale})')

print()
print('§24.1 PASSED -- full Phase 3 forward chain verified with correct shapes.')
print('Proceed to §25 (VLM inference) -> §26 (SAM2) -> §27 (inpainting).')


§24.1  Architecture smoke test (Phase 3)
  [1] VLM hidden state    : (1, 2048)
  [2] VLM projected       : (1, 1, 768)
  [3] CLIP text embeds    : (1, 77, 768)
  [4] Combined cond (CFG) : (2, 78, 768) [uncond||cond]
  [5] Source latents      : (1, 4, 64, 64)
  [6] Mask (latent size)  : (1, 1, 64, 64)
  [7] UNet 9-ch input (x2): (2, 9, 64, 64) (noisy|mask|masked_src)
  [8] Noise prediction    : (2, 4, 64, 64)
  [9] CFG guided pred     : (1, 4, 64, 64) (scale=7.5)

§24.1 PASSED -- full Phase 3 forward chain verified with correct shapes.
Proceed to §25 (VLM inference) -> §26 (SAM2) -> §27 (inpainting).


## §25 — VLM Inference
Two functions operate on each source image:
- `run_vlm_inference()` — runs `generate()` to get the JSON bbox prediction
- `extract_hidden_state_inference()` — runs a forward pass to get the VLM
  hidden state used as conditioning by the SD diffusion model

Both use `build_messages()` with no annotations (Phase 3 distribution gap acknowledged).

In [17]:
# -- §25.1  run_vlm_inference + parse_vlm_output --------------------------------
#
# v3.0 note: the VLM still produces {edit_type, bbox, edit_description}.
# In v3.0 the bbox is treated as a *secondary* signal — preferred only when
# GroundingDINO returns no detections. The subject phrase is extracted from
# edit_description in §25.4 (extract_subject_phrase).

import torch, json as _json_mod, re as _re
from typing import Optional


def parse_vlm_output(generated_text: str) -> Optional[dict]:
    """Parse VLM generated text to extract {edit_type, bbox, edit_description}.

    Args:
        generated_text: raw string from model.generate() + batch_decode()

    Returns:
        dict with keys: edit_type (str), bbox (list[4 int]), edit_description (str)
        or None if parse fails.
    """
    text = generated_text.strip()
    _fence_match = _re.match(r"^```(?:json)?\s*\n?(.*?)\n?```$", text, _re.DOTALL)
    if _fence_match:
        text = _fence_match.group(1).strip()

    _json_match = _re.search(r"\{[^{}]*\}", text, _re.DOTALL)
    if _json_match and _json_match.group(0) != text:
        log.debug(f"parse_vlm_output: extracted JSON object from longer text")
        text = _json_match.group(0)

    try:
        data = _json_mod.loads(text)
    except _json_mod.JSONDecodeError as _e:
        log.warning(f"parse_vlm_output: JSON decode failed: {_e}\nRaw: {generated_text[:300]!r}")
        return None

    if "edit_type" not in data:
        log.warning(f"parse_vlm_output: missing edit_type. Raw: {generated_text[:200]!r}")
        return None
    if "bbox" not in data or not isinstance(data["bbox"], (list, tuple)) or len(data["bbox"]) != 4:
        log.warning(f"parse_vlm_output: invalid bbox field. Raw: {generated_text[:200]!r}")
        return None
    if "edit_description" not in data:
        log.warning(f"parse_vlm_output: missing edit_description. Raw: {generated_text[:200]!r}")
        return None

    try:
        bbox = [int(round(float(v))) for v in data["bbox"]]
    except (TypeError, ValueError) as _e:
        log.warning(f"parse_vlm_output: bbox value conversion failed: {_e}")
        return None

    if any(v < 0 or v > 1000 for v in bbox):
        log.warning(f"parse_vlm_output: bbox {bbox} outside [0,1000]; clipping.")
        bbox = [max(0, min(v, 1000)) for v in bbox]

    x1, y1, x2, y2 = bbox
    if x2 <= x1 or y2 <= y1:
        log.warning(
            f"parse_vlm_output: degenerate bbox {bbox} (zero/negative area). "
            f"Returning None — caller should use fallback."
        )
        return None

    edit_type = str(data["edit_type"]).strip().lower()
    if edit_type not in KNOWN_EDIT_TYPES:
        log.warning(
            f"parse_vlm_output: unknown edit_type {edit_type!r}; "
            f"known: {sorted(KNOWN_EDIT_TYPES)}. Keeping value."
        )

    return {
        "edit_type":        edit_type,
        "bbox":             bbox,
        "edit_description": str(data["edit_description"]).strip(),
    }


def run_vlm_inference(
    source_image,
    instruction: str,
    device: str = None,
) -> dict:
    """Run the fine-tuned VLM to predict edit_type, bbox, and edit_description.

    Returns a dict that always contains a usable bbox (falling back to
    CFG.phase3_fallback_bbox if parsing failed). The bbox here is the *VLM*
    bbox; v3.0 §26.1 will combine it with the grounding-model bbox.
    """
    _device = device or DEVICE

    if isinstance(source_image, (str, Path)):
        source_image = Image.open(source_image).convert("RGB")

    messages = build_messages(source_image, instruction, annotations=None)

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text           = [text_prompt],
        images         = image_inputs if image_inputs else None,
        videos         = video_inputs if video_inputs else None,
        padding        = False,
        return_tensors = "pt",
    )

    _input_ids  = inputs["input_ids"].to(_device)
    _attn_mask  = inputs["attention_mask"].to(_device)
    _extra_kw   = {}
    if "pixel_values" in inputs:
        _extra_kw["pixel_values"]   = inputs["pixel_values"].to(_device)
    if "image_grid_thw" in inputs:
        _extra_kw["image_grid_thw"] = inputs["image_grid_thw"].to(_device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            input_ids      = _input_ids,
            attention_mask = _attn_mask,
            max_new_tokens = CFG.phase3_max_new_tokens,
            do_sample      = False,
            pad_token_id   = vlm_processor.tokenizer.eos_token_id,
            **_extra_kw,
        )

    generated_ids = output_ids[:, _input_ids.shape[1]:]
    raw_text = vlm_processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]

    parsed = parse_vlm_output(raw_text)

    if parsed is not None:
        return {
            "raw_text":         raw_text,
            "parsed":           parsed,
            "edit_type":        parsed["edit_type"],
            "bbox":             parsed["bbox"],
            "edit_description": parsed["edit_description"],
            "used_fallback":    False,
        }
    log.warning(
        f"run_vlm_inference: parse failed for instruction {instruction[:60]!r}. "
        f"Using fallback bbox {list(CFG.phase3_fallback_bbox)}."
    )
    return {
        "raw_text":         raw_text,
        "parsed":           None,
        "edit_type":        "unknown",
        "bbox":             list(CFG.phase3_fallback_bbox),
        "edit_description": instruction,
        "used_fallback":    True,
    }


# -- parse_vlm_output smoke tests -----------------------------------------
_good_json = '{"edit_type": "remove", "bbox": [100, 200, 600, 800], "edit_description": "remove the car"}'
_p = parse_vlm_output(_good_json)
assert _p is not None and _p["bbox"] == [100, 200, 600, 800]

_fenced = '```json\n{"edit_type": "add", "bbox": [0, 0, 500, 500], "edit_description": "add flowers"}\n```'
_p2 = parse_vlm_output(_fenced)
assert _p2 is not None and _p2["edit_type"] == "add"

_bad = "Sorry, I cannot do that."
_p3 = parse_vlm_output(_bad)
assert _p3 is None

_oob = '{"edit_type": "adjust", "bbox": [-10, 0, 1100, 500], "edit_description": "brighten"}'
_p4 = parse_vlm_output(_oob)
assert _p4 is not None and _p4["bbox"] == [0, 0, 1000, 500], f"OOB clip failed: {_p4}"

print("§25.1 run_vlm_inference + parse_vlm_output defined (v3.0).")
print("  ok  parse_vlm_output: valid JSON, fenced JSON, bad text, OOB bbox clipping")


Raw: 'Sorry, I cannot do that.'


§25.1 run_vlm_inference + parse_vlm_output defined (v3.0).
  ok  parse_vlm_output: valid JSON, fenced JSON, bad text, OOB bbox clipping


In [18]:
# -- §25.2  extract_hidden_state_inference -------------------------------------
#
# SPEC REQUIREMENT: 'Qwen hidden-state extraction must exclude image tokens and
# special/template tokens before mean pooling.'
#
# This function is IDENTICAL in logic to Phase 1 §11.1 extract_hidden_state_for_sample,
# but simplified for inference (no sample_meta dict -- takes image + instruction directly).
#
# Key invariants (must match Phase 1 §11.1):
#   - Same build_messages() call (user-turn only, add_generation_prompt=True)
#   - Same QWEN_SPECIAL_START threshold (151643)
#   - Same mean-pool over text tokens from last hidden layer (index -1)
#   - Same output dtype: float32, shape (vlm_hidden_dim,) on CPU
#
# Called once per image, before the DDIM denoising loop.

import torch
from typing import Optional


def extract_hidden_state_inference(
    source_image,
    instruction: str,
    device: str = None,
) -> Optional[torch.Tensor]:
    '''Extract VLM hidden state for SD conditioning (inference-time version).

    Runs a user-turn-only forward pass with output_hidden_states=True.
    Filters to text tokens (ID < QWEN_SPECIAL_START=151643) then mean-pools.
    Output is the same (hidden_dim,) float32 vector used in Phase 2 training,
    but extracted WITHOUT ground-truth annotation regions in the prompt
    (known distribution gap -- see Phase 3 notebook header for discussion).

    Args:
        source_image: PIL.Image.Image or Path/str
        instruction:  edit instruction string
        device:       target device (default DEVICE)

    Returns:
        Tensor (vlm_hidden_dim,) float32 on CPU, or None on failure.
        Returns None (not raises) -- caller should skip sample on failure.
    '''
    _device = device or DEVICE

    try:
        if isinstance(source_image, (str, Path)):
            source_image = Image.open(source_image).convert('RGB')

        # Build user-turn messages (no annotations -- matches Phase 3 generation prompt)
        messages = build_messages(source_image, instruction, annotations=None)

        # Apply chat template (user-turn only, with generation prompt)
        text_prompt = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = vlm_processor(
            text           = [text_prompt],
            images         = image_inputs if image_inputs else None,
            videos         = video_inputs if video_inputs else None,
            padding        = False,
            return_tensors = 'pt',
        )

        input_ids = inputs['input_ids'].to(_device)
        attn_mask = inputs['attention_mask'].to(_device)
        extra = {}
        if 'pixel_values' in inputs:
            extra['pixel_values']   = inputs['pixel_values'].to(_device)
        if 'image_grid_thw' in inputs:
            extra['image_grid_thw'] = inputs['image_grid_thw'].to(_device)

        with torch.no_grad():
            outputs = vlm_model(
                input_ids            = input_ids,
                attention_mask       = attn_mask,
                output_hidden_states = True,
                **extra,
            )

        # Last hidden layer: (1, seq_len, hidden_dim)
        last_hidden = outputs.hidden_states[-1]
        assert last_hidden.ndim == 3 and last_hidden.shape[0] == 1, (
            f'Unexpected last_hidden shape: {last_hidden.shape}'
        )

        id_list = input_ids[0].tolist()
        seq_len = last_hidden.shape[1]
        assert len(id_list) == seq_len, (
            f'id_list len {len(id_list)} != seq_len {seq_len}'
        )

        # Boolean mask: True = text token (keep for mean pool), False = special/image (exclude)
        text_mask = torch.tensor(
            [tok_id < QWEN_SPECIAL_START for tok_id in id_list],
            dtype=torch.bool, device=_device,
        )  # (seq_len,)

        n_text_tokens = text_mask.sum().item()
        if n_text_tokens == 0:
            log.warning(
                f'extract_hidden_state_inference: no text tokens found '
                f'(all {seq_len} token IDs >= QWEN_SPECIAL_START={QWEN_SPECIAL_START}). '
                f'Returning None.'
            )
            return None

        # Mean pool text tokens: (n_text, hidden_dim) -> (hidden_dim,)
        text_hidden = last_hidden[0][text_mask]  # (n_text, hidden_dim)
        pooled      = text_hidden.mean(dim=0)    # (hidden_dim,)

        assert pooled.shape == (CFG.vlm_hidden_dim,), (
            f'Pooled shape {pooled.shape} != ({CFG.vlm_hidden_dim},). '
            f'Check CFG.vlm_hidden_dim matches the loaded model.'
        )

        log.debug(
            f'extract_hidden_state_inference: seq_len={seq_len}, '
            f'text_tokens={n_text_tokens} '
            f'({100*n_text_tokens/seq_len:.1f}%), '
            f'range=[{pooled.min().item():.3f}, {pooled.max().item():.3f}]'
        )

        return pooled.float().cpu()   # float32 on CPU

    except Exception as _e:
        log.warning(f'extract_hidden_state_inference failed: {type(_e).__name__}: {_e}')
        return None


print('§25.2 extract_hidden_state_inference() defined.')
print(f'  QWEN_SPECIAL_START = {QWEN_SPECIAL_START}  (text tokens: ID < threshold)')
print(f'  Output: ({CFG.vlm_hidden_dim},) float32 CPU')
print(f'  Forward: user-turn only (add_generation_prompt=True, no assistant tokens)')
print(f'  Pooling: mean over text tokens, last hidden layer')


§25.2 extract_hidden_state_inference() defined.
  QWEN_SPECIAL_START = 151643  (text tokens: ID < threshold)
  Output: (2048,) float32 CPU
  Forward: user-turn only (add_generation_prompt=True, no assistant tokens)
  Pooling: mean over text tokens, last hidden layer


In [19]:
# -- §25.3  VLM inference smoke test ------------------------------------------
# Tests run_vlm_inference() and extract_hidden_state_inference() on a real sample
# from the filtered manifest. Verifies:
#   - VLM generates valid JSON (or fallback triggers gracefully)
#   - bbox is in [0,1000] range with positive area
#   - hidden state has correct shape, dtype, no NaN

import json as _json_mod

print('=' * 60)
print('§25.3  VLM inference smoke test (3 samples)')
print('=' * 60)

_manifest = load_json(CFG.filtered_manifest_path)
# Pick samples with shard_id (Phase 1 extraction succeeded) for realistic test
_smoke_samples = [m for m in _manifest if m.get('shard_id') is not None][:3]
if not _smoke_samples:
    print('  WARN  No shard_id samples in manifest -- using first 3 samples')
    _smoke_samples = _manifest[:3]

for _i, _entry in enumerate(_smoke_samples):
    _src_path = CFG.data_dir / _entry['source_image']
    _instr    = _entry.get('edit_instruction', 'edit the image')

    print(f'\nSample {_i+1}: {_entry["sample_id"]}')
    print(f'  Instruction: {_instr[:70]!r}')

    if not _src_path.exists():
        print(f'  SKIP  source image not found: {_src_path}')
        continue

    # Test VLM generation
    _result = run_vlm_inference(_src_path, _instr)
    print(f'  Raw output: {_result["raw_text"][:100]!r}')
    if _result['used_fallback']:
        print(f'  WARN  Parse failed -- fallback bbox used: {_result["bbox"]}')
    else:
        _bbox = _result['bbox']
        assert all(0 <= v <= 1000 for v in _bbox), f'bbox out of range: {_bbox}'
        x1, y1, x2, y2 = _bbox
        assert x2 > x1 and y2 > y1, f'degenerate bbox: {_bbox}'
        print(f'  ok  bbox={_bbox}  edit_type={_result["edit_type"]!r}')

    # Test hidden state extraction
    _hs = extract_hidden_state_inference(_src_path, _instr)
    if _hs is None:
        print(f'  FAIL  Hidden state extraction returned None')
    else:
        assert _hs.shape == (CFG.vlm_hidden_dim,), f'HS shape {_hs.shape}'
        assert not _hs.isnan().any(), 'NaN in hidden state'
        assert _hs.dtype == torch.float32
        print(f'  ok  hidden state: shape={tuple(_hs.shape)}, '
              f'range=[{_hs.min().item():.3f}, {_hs.max().item():.3f}]')

print('\n§25.3 VLM inference smoke test complete.')


§25.3  VLM inference smoke test (3 samples)

Sample 1: 0000001
  Instruction: 'Change vest positioned in the right-central area to red plaid'


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Raw output: '{"edit_type": "adjust", "bbox": [489, 375, 569, 513], "edit_description": "Change vest positioned in'
  ok  bbox=[489, 375, 569, 513]  edit_type='adjust'
  ok  hidden state: shape=(2048,), range=[-58.250, 37.500]

Sample 2: 0000002
  Instruction: 'Turn bridge positioned in the upper-central area into red'


  Raw output: '{"edit_type": "adjust", "bbox": [263, 134, 1091, 435], "edit_description": "Turn bridge positioned i'
  ok  bbox=[263, 134, 1000, 435]  edit_type='adjust'
  ok  hidden state: shape=(2048,), range=[-58.500, 37.750]

Sample 3: 0000003
  Instruction: 'Turn cliffs positioned in the upper-right area into darker gray'


  Raw output: '{"edit_type": "adjust", "bbox": [345, 0, 1091, 687], "edit_description": "Turn cliffs positioned in '
  ok  bbox=[345, 0, 1000, 687]  edit_type='adjust'
  ok  hidden state: shape=(2048,), range=[-58.500, 37.500]

§25.3 VLM inference smoke test complete.


## §25.4 — Subject Phrase Extraction (v3.0)

In [20]:
# -- §25.4  extract_subject_phrase  (v3.0) -----------------------------------
#
# Goal: produce a short noun phrase suitable for GroundingDINO from
# (a) the VLM's edit_description and (b) the original NL instruction.
#
# Strategy (rule-based — deliberate, no extra model required):
#   1. Strip common edit-action verbs/prefixes that won't help grounding
#      ("turn the X into ...", "change X to ...", "replace X with ...",
#       "remove X", "add X", "make the X ...").
#   2. After stripping, take the FIRST 1–4 word noun-phrase chunk.
#   3. Drop trailing modifiers introduced by " into "/" to "/" with " — these
#      describe the *target* state (red, white crust), not what to localize.
#   4. Fall back to the noun head extracted from the original instruction if
#      the description is empty or all-verb.
#
# This is intentionally not perfect — GroundingDINO is forgiving about extra
# words ("motorcycle on the asphalt" still detects motorcycles), so we err on
# the side of keeping more content rather than over-pruning. If the rule
# returns garbage the §26.1 pipeline still falls back to the VLM bbox.

import re as _re_subj

# Verbs/phrases that reliably precede the *subject* in ImgEdit-style instructions.
# Order matters: we strip from the longest pattern first.
_LEADING_PATTERNS = [
    r"^turn\s+(?:the\s+|a\s+|an\s+)?",
    r"^change\s+(?:the\s+|a\s+|an\s+)?",
    r"^replace\s+(?:the\s+|a\s+|an\s+)?",
    r"^remove\s+(?:the\s+|a\s+|an\s+)?",
    r"^delete\s+(?:the\s+|a\s+|an\s+)?",
    r"^add\s+(?:a\s+|an\s+|the\s+)?",
    r"^make\s+(?:the\s+|a\s+|an\s+)?",
    r"^edit\s+(?:the\s+|a\s+|an\s+)?",
    r"^modify\s+(?:the\s+|a\s+|an\s+)?",
    r"^paint\s+(?:the\s+|a\s+|an\s+)?",
    r"^recolour\s+(?:the\s+|a\s+|an\s+)?",
    r"^recolor\s+(?:the\s+|a\s+|an\s+)?",
    r"^transform\s+(?:the\s+|a\s+|an\s+)?",
]

# Trailing-modifier markers — anything to their RIGHT describes the target
# state, not the object to localize.
_TARGET_MARKERS = (
    " into ", " to a ", " to an ", " to the ", " with ", " using ",
    " so that ", " so it ", " in a ", " in the style of ",
)

# Localization-noise tokens that often appear in VLM descriptions and that
# we should drop when they are the *trailing* word (positional fluff).
_TRAILING_NOISE = {
    "area", "region", "section", "part", "side",
    "image", "picture", "scene", "frame", "centre", "center",
    "background", "foreground",
}

# Filler positional phrases — strip these mid-string. Order matters.
_POSITIONAL_FRAGMENTS = [
    "positioned in the upper-right area",
    "positioned in the upper-left area",
    "positioned in the lower-right area",
    "positioned in the lower-left area",
    "positioned in the central area",
    "positioned in the upper area",
    "positioned in the lower area",
    "positioned in the right area",
    "positioned in the left area",
    "positioned in the centre area",
    "positioned in the center area",
    "positioned at the upper right",
    "positioned at the upper left",
    "positioned at the lower right",
    "positioned at the lower left",
    "in the upper-right area",
    "in the upper-left area",
    "in the lower-right area",
    "in the lower-left area",
    "in the central area",
    "in the centre",
    "in the center",
]


def extract_subject_phrase(
    edit_description: str,
    instruction: str = "",
) -> str:
    """Extract a short noun phrase suitable as a GroundingDINO prompt.

    Args:
        edit_description: from the VLM's JSON output.
        instruction: original NL instruction (used as fallback).

    Returns:
        A lowercased noun phrase, or "" if extraction failed.
    """
    candidates = [s for s in (edit_description or "", instruction or "") if s.strip()]
    for raw in candidates:
        text = raw.strip().lower()

        # 1. Strip filler positional fragments first (so they don't interfere
        #    with leading-pattern matching).
        for frag in _POSITIONAL_FRAGMENTS:
            text = text.replace(frag, " ")
        text = _re_subj.sub(r"\s+", " ", text).strip()

        # 2. Strip leading edit-action prefix.
        for pat in _LEADING_PATTERNS:
            new = _re_subj.sub(pat, "", text)
            if new != text:
                text = new.strip()
                break

        # 3. Truncate at the first target marker.
        lower = text.lower()
        cut = len(lower)
        for marker in _TARGET_MARKERS:
            i = lower.find(marker)
            if i >= 0 and i < cut:
                cut = i
        text = text[:cut].strip()

        # 4. Drop trailing punctuation.
        text = text.rstrip(".,;:!?- ").strip()

        # 5. Drop trailing positional-noise words.
        toks = text.split()
        while toks and toks[-1] in _TRAILING_NOISE:
            toks.pop()
        text = " ".join(toks)

        # 6. Cap to first 5 tokens — GroundingDINO loses precision on very
        #    long noun-phrase prompts.
        toks = text.split()
        if len(toks) > 5:
            toks = toks[:5]
        text = " ".join(toks).strip()

        if text:
            return text

    return ""


# -- Smoke tests -----------------------------------------------------------
_t = extract_subject_phrase("Turn motorcycle positioned in the central area into red")
assert _t == "motorcycle", f"motorcycle test failed: {_t!r}"

_t = extract_subject_phrase("Turn bread positioned in the upper-right area into white crust")
assert _t == "bread", f"bread test failed: {_t!r}"

_t = extract_subject_phrase("Turn text positioned in the central area into red festive font")
assert _t == "text", f"text test failed: {_t!r}"

_t = extract_subject_phrase("Replace the dog with a cat")
assert _t == "dog", f"replace-dog test failed: {_t!r}"

_t = extract_subject_phrase("Change shirt colour to blue")
assert _t.startswith("shirt"), f"shirt test failed: {_t!r}"

_t = extract_subject_phrase("Remove the car from the parking lot")
assert _t.startswith("car"), f"car test failed: {_t!r}"

_t = extract_subject_phrase("", "remove the bicycle")
assert _t.startswith("bicycle"), f"fallback-instruction test failed: {_t!r}"

_t = extract_subject_phrase("")
assert _t == "", f"empty test failed: {_t!r}"

print("§25.4 extract_subject_phrase defined (v3.0).")
print("  ok  motorcycle / bread / text / dog / shirt / car / fallback / empty")


§25.4 extract_subject_phrase defined (v3.0).
  ok  motorcycle / bread / text / dog / shirt / car / fallback / empty


## §26 — SAM2 Mask Generation

`bbox_to_sam2_mask(source_image, bbox_rel, device)` converts a VLM-predicted
relative bbox ([0,1000] xyxy) to a binary uint8 mask via SAM2.

**Global edit handling:** if the bbox covers >90% of the image area (or is the
full 0,0,1000,1000 fallback), SAM2 is skipped and an all-ones mask is returned.
This avoids prompting SAM2 with a degenerate box that spans the whole image.

**Coordinate conversion:** VLM coords are in [0,1000]; SAM2 needs absolute pixels.
`bbox_relative_to_absolute(bbox, H, W)` maps [0,1000] → [0,W] / [0,H] correctly.

**Output:** uint8 ndarray (H,W) with values 0/255 (SAM2 convention: 255=mask).


In [21]:
# -- §26.1  bbox_to_sam2_mask  (v3.0 — grounding-model + multi-mask + rerank)
#
# Pipeline (per call):
#   1.  Build a subject phrase via §25.4 extract_subject_phrase().
#   2.  Run GroundingDINO with that phrase. Returns N candidate (bbox, score).
#   3.  Sanity-check the VLM bbox (degenerate / out-of-bounds detection).
#   4.  Pick the bbox to feed SAM2:
#         - GroundingDINO top-score box if available.
#         - Else fall back to VLM bbox (provided it is non-degenerate).
#         - Else fall back to the global-edit short-circuit (full-image mask).
#   5.  Run SAM2 with multimask_output=True + (box, optional centre point).
#       Returns up to 3 candidate masks.
#   6.  Rerank candidates by CLIP image-text similarity to the subject phrase.
#       (When CLIP rerank is disabled the highest-score SAM2 mask is used.)
#   7.  Mask post-processing: morphological closing (kernel=CFG.mask_morph_close_kernel)
#       and largest-connected-component selection.
#
# All decisions and per-step diagnostics are returned via the `diagnostics`
# dict alongside the mask, so the batch-inference loop can persist them for
# later analysis.

import numpy as np
import torch
import torch.nn.functional as _F
from PIL import Image
from typing import Optional, Tuple, List


def _bbox_xyxy_to_centre_point(bbox_abs: List[int]) -> Tuple[int, int]:
    x1, y1, x2, y2 = bbox_abs
    return (int(round((x1 + x2) / 2)), int(round((y1 + y2) / 2)))


def _bbox_area_frac(bbox_abs: List[int], W: int, H: int) -> float:
    x1, y1, x2, y2 = bbox_abs
    return max(0.0, (x2 - x1) * (y2 - y1)) / max(1.0, float(W * H))


def _is_vlm_bbox_degenerate(bbox_rel: List[int]) -> bool:
    """Detect VLM bboxes that we should not trust spatially.

    Returns True for:
      - Full-image / near-full-image boxes (>= bbox_degenerate_max_frac).
      - Pinpoint / tiny boxes (<= bbox_degenerate_min_frac).
      - Boxes flagged by is_full_image_bbox (the [0,0,1000,1000] fallback).
    """
    if is_full_image_bbox(bbox_rel):
        return True
    x1, y1, x2, y2 = bbox_rel
    rel_area = max(0, x2 - x1) * max(0, y2 - y1) / 1_000_000.0
    if rel_area >= CFG.bbox_degenerate_max_frac:
        return True
    if rel_area <= CFG.bbox_degenerate_min_frac:
        return True
    return False


def _mask_postprocess(mask_uint8: np.ndarray) -> np.ndarray:
    """Morphological closing + largest-connected-component selection.

    Args:
        mask_uint8: (H, W) uint8 mask, 0 = bg, 255 = fg.

    Returns:
        Cleaned (H, W) uint8 mask with the same polarity.
    """
    H, W = mask_uint8.shape
    bin_mask = mask_uint8 > 127

    # 1. Morphological closing fills small holes / thin gaps.
    if CFG.mask_morph_close_kernel and CFG.mask_morph_close_kernel > 1:
        k = int(CFG.mask_morph_close_kernel)
        bin_mask = _scipy_ndimage.binary_closing(
            bin_mask, structure=np.ones((k, k), dtype=bool)
        )

    # 2. Largest connected component (drop speckle).
    if CFG.mask_keep_largest_component:
        labels, n_cc = _scipy_ndimage.label(bin_mask)
        if n_cc > 1:
            sizes = _scipy_ndimage.sum(bin_mask, labels, range(1, n_cc + 1))
            largest = int(np.argmax(sizes)) + 1
            bin_mask = labels == largest
        elif n_cc == 0:
            return np.zeros_like(mask_uint8)

    # 3. Drop the whole mask if it is below the minimum area fraction.
    if bin_mask.mean() < CFG.mask_min_component_area_frac:
        log.warning(
            "mask_postprocess: mask area %.4f < min %.4f — returning original mask",
            bin_mask.mean(), CFG.mask_min_component_area_frac,
        )
        return mask_uint8

    return (bin_mask.astype(np.uint8) * 255)


def _clip_rerank_masks(
    image_pil: Image.Image,
    masks: List[np.ndarray],
    text_phrase: str,
) -> int:
    """Pick the index of the mask whose masked image best matches `text_phrase`
    in CLIP image-text similarity space.

    We reuse the CLIPTextModel + tokenizer that SD already loaded for text
    conditioning. Image features come from the SD VAE-paired CLIP-vision encoder
    that we DON'T have on hand, so we use a lightweight proxy: average pixel
    embedding via the SD CLIP text branch is not appropriate. Instead we use
    transformers.CLIPModel (vit-base-patch32) which we lazily load once and cache.
    """
    global _clip_rerank_model, _clip_rerank_processor
    try:
        _ = _clip_rerank_model
    except NameError:
        from transformers import CLIPModel, CLIPProcessor
        _CLIP_RERANK_ID = "openai/clip-vit-base-patch32"
        log.info(f"_clip_rerank: lazy-loading {_CLIP_RERANK_ID} for mask rerank")
        _clip_rerank_processor = CLIPProcessor.from_pretrained(_CLIP_RERANK_ID)
        _clip_rerank_model = CLIPModel.from_pretrained(_CLIP_RERANK_ID).to(DEVICE).eval()
        for _p in _clip_rerank_model.parameters():
            _p.requires_grad_(False)

    arr = np.array(image_pil.convert("RGB"))
    masked_imgs = []
    for m in masks:
        # Build a 3-channel masked image: mask region kept, background blacked.
        # CLIP's training distribution doesn't see hard cuts cleanly, so we also
        # tightly crop to the mask bbox to give CLIP a natural-looking patch.
        ys, xs = np.where(m > 127)
        if ys.size == 0 or xs.size == 0:
            # Empty mask — give CLIP the whole image so it still gets *some* score.
            masked_imgs.append(image_pil)
            continue
        y1, y2 = int(ys.min()), int(ys.max()) + 1
        x1, x2 = int(xs.min()), int(xs.max()) + 1
        crop = arr[y1:y2, x1:x2]
        masked_imgs.append(Image.fromarray(crop))

    inputs = _clip_rerank_processor(
        text=[text_phrase] * len(masked_imgs),
        images=masked_imgs,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        out = _clip_rerank_model(**inputs)
    sims = out.logits_per_image.diag().cpu().numpy()  # one per (image, text) pair
    best = int(np.argmax(sims))
    log.info(f"_clip_rerank: similarities={sims.tolist()} -> picked {best}")
    return best


def bbox_to_sam2_mask(
    source_image: Image.Image,
    bbox_rel: list,
    edit_description: str = "",
    instruction: str = "",
    device=None,
) -> Tuple[np.ndarray, dict]:
    """v3.0 mask generation: grounding-model + SAM2 multi-mask + rerank.

    Args:
        source_image: PIL Image (any mode, will be converted to RGB).
        bbox_rel: VLM-predicted [x1,y1,x2,y2] in [0,1000] relative coords.
        edit_description: from VLM JSON output (used to derive subject phrase).
        instruction: original NL edit instruction (fallback for subject extraction).
        device: torch device (kept for API symmetry; not used here).

    Returns:
        (mask_uint8, diagnostics)
            mask_uint8: (H, W) uint8 mask with 0=preserve, 255=inpaint.
            diagnostics: dict with per-step info for the batch loop.
    """
    assert sam2_predictor is not None, "sam2_predictor not loaded — run §21.2"
    assert grounding_model is not None, "grounding model not loaded — run §22.5"

    diag = {
        "vlm_bbox_rel": list(bbox_rel),
        "vlm_bbox_degenerate": False,
        "subject_phrase": "",
        "grounding_n_boxes": 0,
        "grounding_top_score": None,
        "grounding_top_bbox_abs": None,
        "selected_source": None,            # 'grounding' | 'vlm' | 'global_edit'
        "selected_bbox_abs": None,
        "sam2_n_masks": 0,
        "sam2_scores": [],
        "rerank_used": False,
        "rerank_picked": None,
        "post_clean_kept_largest": CFG.mask_keep_largest_component,
        "final_mask_coverage": None,
    }

    img_rgb = source_image.convert("RGB")
    W, H = img_rgb.size
    img_area = W * H

    # ── Step 1: subject phrase ───────────────────────────────────────────
    subject_phrase = extract_subject_phrase(
        edit_description=edit_description, instruction=instruction
    )
    diag["subject_phrase"] = subject_phrase

    # ── Step 2: GroundingDINO detection ──────────────────────────────────
    grounding_boxes_abs = []   # list of (bbox_xyxy_abs, score)
    if subject_phrase:
        grounding_boxes_abs = run_grounding_dino(
            image=img_rgb, phrase=subject_phrase
        )
        diag["grounding_n_boxes"] = len(grounding_boxes_abs)
        if grounding_boxes_abs:
            top_box, top_score = grounding_boxes_abs[0]
            diag["grounding_top_score"] = float(top_score)
            diag["grounding_top_bbox_abs"] = list(map(int, top_box))

    # ── Step 3: VLM-bbox sanity check ────────────────────────────────────
    diag["vlm_bbox_degenerate"] = _is_vlm_bbox_degenerate(bbox_rel)

    # ── Step 4: select bbox to drive SAM2 ─────────────────────────────────
    selected_bbox_abs = None
    selected_source = None

    if grounding_boxes_abs:
        selected_bbox_abs = list(map(int, grounding_boxes_abs[0][0]))
        selected_source = "grounding"
    elif not diag["vlm_bbox_degenerate"]:
        x1_a, y1_a, x2_a, y2_a = bbox_relative_to_absolute(bbox_rel, W, H)
        selected_bbox_abs = [x1_a, y1_a, x2_a, y2_a]
        selected_source = "vlm"
    else:
        # No reliable spatial signal — fall through to global-edit branch below.
        selected_source = "global_edit"

    # Global-edit short-circuit (covers both the explicit case and the fallback
    # path above). Returns a uniform full-image mask without invoking SAM2.
    if (
        selected_source == "global_edit"
        or (
            selected_bbox_abs is not None
            and _bbox_area_frac(selected_bbox_abs, W, H) >= CFG.global_edit_area_frac
        )
    ):
        log.info(
            "bbox_to_sam2_mask: global-edit short-circuit fired (source=%s) — "
            "returning full-image mask.", selected_source,
        )
        diag["selected_source"] = "global_edit"
        diag["selected_bbox_abs"] = [0, 0, W, H]
        full_mask = np.full((H, W), 255, dtype=np.uint8)
        diag["final_mask_coverage"] = 1.0
        return full_mask, diag

    # Clamp the chosen bbox to image bounds.
    x1_a, y1_a, x2_a, y2_a = selected_bbox_abs
    x1_a = max(0, min(x1_a, W - 1))
    y1_a = max(0, min(y1_a, H - 1))
    x2_a = max(x1_a + 1, min(x2_a, W))
    y2_a = max(y1_a + 1, min(y2_a, H))
    selected_bbox_abs = [x1_a, y1_a, x2_a, y2_a]
    diag["selected_source"] = selected_source
    diag["selected_bbox_abs"] = selected_bbox_abs

    # ── Step 5: SAM2 multi-mask ──────────────────────────────────────────
    box_np = np.array([selected_bbox_abs], dtype=np.float32)
    point_coords = None
    point_labels = None
    if CFG.sam2_use_centre_point:
        cx, cy = _bbox_xyxy_to_centre_point(selected_bbox_abs)
        point_coords = np.array([[cx, cy]], dtype=np.float32)
        point_labels = np.array([1], dtype=np.int32)   # 1 = positive

    sam2_predictor.set_image(np.array(img_rgb))
    masks, scores, _logits = sam2_predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        box=box_np,
        multimask_output=CFG.sam2_multimask_output,
    )
    # masks shape:
    #   multimask=True  -> (3, H, W) bool
    #   multimask=False -> (1, H, W) bool
    diag["sam2_n_masks"] = int(masks.shape[0])
    diag["sam2_scores"]  = [float(s) for s in scores.tolist()]

    masks_uint8 = [(m.astype(np.uint8) * 255) for m in masks]

    # ── Step 6: rerank ────────────────────────────────────────────────────
    if (
        CFG.sam2_clip_rerank
        and len(masks_uint8) > 1
        and subject_phrase
    ):
        try:
            best_idx = _clip_rerank_masks(img_rgb, masks_uint8, subject_phrase)
            diag["rerank_used"]  = True
            diag["rerank_picked"] = int(best_idx)
        except Exception as _e:
            log.warning(f"_clip_rerank_masks failed: {_e!r} — falling back to top SAM2 score")
            best_idx = int(np.argmax(scores))
    else:
        best_idx = int(np.argmax(scores))
    chosen_mask = masks_uint8[best_idx]

    # ── Step 7: post-process ─────────────────────────────────────────────
    cleaned = _mask_postprocess(chosen_mask)
    cov = float((cleaned > 127).mean())
    diag["final_mask_coverage"] = cov

    log.info(
        "bbox_to_sam2_mask v3: source=%s subject=%r grounding_n=%d sam_scores=%s "
        "picked=%d coverage=%.3f",
        diag["selected_source"], subject_phrase, diag["grounding_n_boxes"],
        diag["sam2_scores"], best_idx, cov,
    )

    assert cleaned.shape == (H, W), (
        f"Mask shape {cleaned.shape} != image shape ({H},{W})"
    )
    return cleaned, diag


print("bbox_to_sam2_mask defined (v3.0 — grounding + multi-mask + rerank).")


bbox_to_sam2_mask defined (v3.0 — grounding + multi-mask + rerank).


In [22]:
# -- §26.2  SAM2 mask smoke test (v3.0)  ------------------------------------
# Uses the first real manifest sample to verify the new bbox_to_sam2_mask
# return signature: (mask_uint8, diagnostics_dict).

assert sam2_predictor is not None, "Load SAM2 first (§21.2)"
assert grounding_model is not None, "Load GroundingDINO first (§22.5)"
assert len(manifest_phase3) > 0, "No manifest samples — run §20.2"

sample = manifest_phase3[0]
img_path = CFG.data_dir / sample["source_image"]
assert img_path.exists(), f"Image not found: {img_path}"
source_image = Image.open(img_path).convert("RGB")
W_img, H_img = source_image.size

ann = sample.get("annotations", [{}])[0]
bbox_rel = ann.get("bbox", [200, 200, 600, 600])
instr    = sample.get("edit_instruction", "edit the main object")
print(f"Test sample: {sample['source_image']}  bbox_rel={bbox_rel}")
print(f"Image size: {W_img}x{H_img}")
print(f"Instruction: {instr!r}")

mask_out, diag = bbox_to_sam2_mask(
    source_image=source_image,
    bbox_rel=bbox_rel,
    edit_description=instr,
    instruction=instr,
)

assert mask_out.shape == (H_img, W_img), (
    f"Expected ({H_img},{W_img}), got {mask_out.shape}"
)
assert mask_out.dtype == np.uint8, f"dtype {mask_out.dtype} != uint8"
print(f"Mask coverage: {diag['final_mask_coverage']*100:.1f}%")
print(f"Selected source: {diag['selected_source']}  subject={diag['subject_phrase']!r}")
print(f"Grounding n_boxes={diag['grounding_n_boxes']}  top_score={diag['grounding_top_score']}")
print(f"SAM2 candidate scores: {diag['sam2_scores']}  rerank_picked={diag['rerank_picked']}")
print("SAM2 v3.0 mask smoke test PASSED.")


Test sample: images/0000001_source.jpg  bbox_rel=[200, 200, 600, 600]
Image size: 1535x1024
Instruction: 'Change vest positioned in the right-central area to red plaid'


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Mask coverage: 1.1%
Selected source: grounding  subject='vest positioned in the right-central'
Grounding n_boxes=3  top_score=0.604024350643158
SAM2 candidate scores: [0.9549944400787354, 0.745046079158783, 0.9650596380233765]  rerank_picked=2
SAM2 v3.0 mask smoke test PASSED.


## §27 — SD Inpainting with CFG

`run_inpainting()` implements the full DDIM denoising loop with Classifier-Free
Guidance (CFG) using the custom 9-channel UNet.

**Input channels:** UNet expects (4+1+4=9) channels:
`[noisy_latent(4) | mask_latent(1) | masked_source_latent(4)]`

**CFG:** Two batch forward passes (uncond ∥ cond). The conditioning tensor
has shape (2B, 78, 768) with uncond first, cond second.

**Mask convention:**
- mask=1 → **inpaint** this region (noisy latent; model generates content)
- mask=0 → **preserve** this region (use source latent)

SAM2 returns 255=foreground; we normalise to [0,1] float mask for the UNet.

**Precision:** VAE encode/decode runs in fp32 to avoid numerical issues;
UNet runs in fp16 (matching Phase 2 training).


In [23]:
# -- §27.1  run_inpainting --------------------------------------------------
#
# SPEC REQUIREMENT: '9-channel UNet input: noisy_target(4) | mask(1) | masked_source(4)'
# SPEC REQUIREMENT: 'Guidance behavior must be handled correctly when passing prompt_embeds'
# SPEC REQUIREMENT: 'DDIM scheduler, 20 inference steps'
#
# Mask polarity (critical -- easy to get wrong):
#   mask = 1 where we INPAINT (replace/edit region)
#   mask = 0 where we PRESERVE (keep source pixels)
#   SAM2 output: 255 in the target region -> normalise to [0,1].
#   masked_source_latent encodes the source image with the edit region ZEROED OUT.
#
# CFG shape guard:
#   build_combined_conditioning(do_cfg=True) returns (2, 78, 768).
#   We tile the latent/mask inputs to batch=2 to match.
#   After UNet forward: split noise_pred into [uncond, cond] along dim=0.
#   CFG formula: noise = uncond + guidance_scale * (cond - uncond)

import torch
import numpy as np
from PIL import Image
from typing import Optional

def run_inpainting(
    source_image: Image.Image,
    vlm_hidden: torch.Tensor,
    instruction: str,
    mask_np: np.ndarray,
    device=None,
    seed: Optional[int] = None,
) -> Image.Image:
    """Run SD 1.5 inpainting with CFG on source_image in the mask region.

    Args:
        source_image: PIL Image (RGB).
        vlm_hidden:   float32 Tensor of shape (vlm_hidden_dim,) on CPU.
                      From extract_hidden_state_inference().
        instruction:  text instruction for CLIP conditioning.
        mask_np:      uint8 ndarray (H,W) with 255=edit region, 0=preserve.
        device:       torch.device. Defaults to DEVICE.
        seed:         Optional int for reproducible inference.

    Returns:
        Edited PIL Image (RGB, same size as source_image).
    """
    if device is None:
        device = DEVICE

    assert vae is not None and unet is not None, "Load SD components first (§23)"
    assert vlm_adapter is not None, "Load VLMProjectionAdapter first (§23.2)"

    H_orig, W_orig = source_image.size[1], source_image.size[0]
    target_size = CFG.phase3_resolution  # 512

    # -- 1. Resize image and mask to 512x512 --------------------------------
    img_resized = source_image.convert('RGB').resize(
        (target_size, target_size), Image.LANCZOS
    )
    # Resize mask: nearest-neighbour to preserve binary values.
    mask_pil = Image.fromarray(mask_np).resize(
        (target_size, target_size), Image.NEAREST
    )
    mask_512 = np.array(mask_pil)

    # -- 2. Build float mask tensor (1,1,H,W) --------------------------------
    # 1 = inpaint, 0 = preserve. SAM2 uses 255=foreground -> divide by 255.
    mask_t = torch.from_numpy(mask_512.astype(np.float32) / 255.0)
    mask_t = mask_t.unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)

    # -- 3. VAE encode source image (fp32) -----------------------------------
    img_arr = np.array(img_resized).astype(np.float32) / 127.5 - 1.0
    img_t = torch.from_numpy(img_arr).permute(2, 0, 1).unsqueeze(0).to(device)  # (1,3,H,W)

    vae.to(device)
    with torch.no_grad():
        # encode in fp32; scale by VAE_SCALE_FACTOR (0.18215)
        vae_fp32 = vae.float()
        source_latent = vae_fp32.encode(img_t).latent_dist.sample()
        source_latent = source_latent * CFG.vae_scale_factor  # (1,4,H/8,W/8)

    latent_H, latent_W = source_latent.shape[2], source_latent.shape[3]
    # shape guard
    assert source_latent.shape == (1, 4, latent_H, latent_W), (
        f"source_latent shape {source_latent.shape} unexpected"
    )

    # -- 4. Build masked source latent ---------------------------------------
    # Resize mask to latent spatial dims (H/8, W/8) for masking source latent.
    mask_latent_np = (mask_512.astype(np.float32) / 255.0)
    mask_latent_pil = Image.fromarray((mask_latent_np * 255).astype(np.uint8)).resize(
        (latent_W, latent_H), Image.NEAREST
    )
    mask_for_latent = torch.from_numpy(
        np.array(mask_latent_pil).astype(np.float32) / 255.0
    ).to(device).unsqueeze(0).unsqueeze(0)  # (1,1,latent_H,latent_W)

    # Zero out source latent in the edit region -> masked_source_latent
    masked_source_latent = source_latent * (1.0 - mask_for_latent)  # (1,4,Hl,Wl)

    # UNet-sized mask: (1,1,latent_H,latent_W)
    mask_for_unet = mask_for_latent  # already at latent spatial size

    # -- 5. Initialize noisy latent ------------------------------------------
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)
    else:
        generator = None

    noise_scheduler.set_timesteps(CFG.phase3_num_inference_steps)
    timesteps = noise_scheduler.timesteps

    init_noise = torch.randn(
        (1, 4, latent_H, latent_W),
        device=device, dtype=torch.float32,
        generator=generator,
    )
    # Scale initial noise by scheduler's sigma (DDIM convention).
    latent = init_noise * noise_scheduler.init_noise_sigma

    # -- 6. Build combined conditioning (CLIP + VLM projection) -------------
    # do_cfg=True -> returns (2*batch, 78, 768): rows 0..B-1 = uncond,
    # rows B..2B-1 = cond. With batch=1, that is (2, 78, 768).
    # extract_hidden_state_inference() returns shape (vlm_hidden_dim,) on CPU;
    # build_combined_conditioning expects (batch, vlm_hidden_dim), so we add
    # the batch dim via unsqueeze(0). The keyword is `vlm_hiddens` (plural) --
    # this must stay byte-identical to Phase 2 §16; do NOT rename the function.
    if vlm_hidden.dim() == 1:
        vlm_hiddens_b = vlm_hidden.unsqueeze(0)  # (1, vlm_hidden_dim)
    else:
        vlm_hiddens_b = vlm_hidden
    assert vlm_hiddens_b.shape == (1, CFG.vlm_hidden_dim), (
        f'vlm_hiddens batched shape {tuple(vlm_hiddens_b.shape)} != (1, {CFG.vlm_hidden_dim})'
    )
    encoder_hidden_states = build_combined_conditioning(
        instructions=[instruction],
        vlm_hiddens=vlm_hiddens_b,
        device=device,
        do_cfg=True,
    )  # (2, 78, 768)
    assert encoder_hidden_states.shape[0] == 2, (
        f"CFG conditioning must have batch=2, got {encoder_hidden_states.shape}"
    )
    assert encoder_hidden_states.shape[1] == 78, (
        f"Conditioning seq_len must be 78 (77 CLIP + 1 VLM), got shape {encoder_hidden_states.shape}"
    )

    # Cast conditioning to fp16 for UNet.
    encoder_hidden_states = encoder_hidden_states.to(dtype=torch.float16)

    # Cast SD components to fp16.
    unet.to(device=device, dtype=torch.float16)

    # -- 7. DDIM denoising loop ----------------------------------------------
    latent = latent.to(dtype=torch.float16)
    source_latent = source_latent.to(dtype=torch.float16)
    masked_source_latent = masked_source_latent.to(dtype=torch.float16)
    mask_for_unet = mask_for_unet.to(dtype=torch.float16)

    unet.eval()
    with torch.no_grad():
        for t in timesteps:
            # Tile latent and mask inputs for CFG batch=2.
            latent_model_input = torch.cat([latent, latent], dim=0)  # (2,4,Hl,Wl)
            mask_input = torch.cat([mask_for_unet, mask_for_unet], dim=0)  # (2,1,Hl,Wl)
            masked_src_input = torch.cat(
                [masked_source_latent, masked_source_latent], dim=0
            )  # (2,4,Hl,Wl)

            # Concatenate along channel dim: (2, 4+1+4, Hl, Wl) = (2,9,Hl,Wl)
            unet_input = torch.cat(
                [latent_model_input, mask_input, masked_src_input], dim=1
            )  # (2, 9, Hl, Wl)

            assert unet_input.shape[1] == 9, (
                f"UNet input must have 9 channels, got {unet_input.shape[1]}"
            )

            # Scale input for DDIM.
            unet_input = noise_scheduler.scale_model_input(unet_input, t)

            # UNet forward: predict noise.
            noise_pred = unet(
                unet_input,
                t,
                encoder_hidden_states=encoder_hidden_states,
            ).sample  # (2, 4, Hl, Wl)

            # Apply CFG: split uncond and cond predictions.
            noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2, dim=0)
            noise_pred_cfg = noise_pred_uncond + CFG.phase3_guidance_scale * (
                noise_pred_cond - noise_pred_uncond
            )

            # Cast to float32 for scheduler step (avoids fp16 overflow in scheduler).
            latent = noise_scheduler.step(
                noise_pred_cfg.float(), t, latent.float()
            ).prev_sample.to(dtype=torch.float16)

    # -- 8. VAE decode -------------------------------------------------------
    with torch.no_grad():
        # Unscale latent before decoding.
        latent_fp32 = latent.float() / CFG.vae_scale_factor
        decoded = vae_fp32.decode(latent_fp32).sample  # (1, 3, H, W) in [-1,1]

    # Convert to uint8 PIL image.
    decoded_np = decoded[0].permute(1, 2, 0).cpu().numpy()
    decoded_np = np.clip((decoded_np + 1.0) * 127.5, 0, 255).astype(np.uint8)
    # NOTE: resize is now handled inside the compositing block below.
    # (We keep decoded_np as 512x512 for compositing, then resize the composite.)

    # -- 9. Composite: paste generated pixels onto source for preserved regions --
    # WHY: The DDIM loop fully regenerates the entire latent from noise, so the
    # decoded image replaces all pixels -- including ones the mask says to preserve.
    # A well-trained model learns to reconstruct background faithfully, but during
    # early training (few epochs) it hallucinates the background, causing the
    # "full-image distortion" artifact. Hard-compositing the source pixels back into
    # the preserve region (mask=0) guarantees zero background drift regardless of
    # training maturity, at no quality cost in the edit region.
    #
    # PIL.Image.composite(foreground, background, mask):
    #   mask white (255) -> use foreground (generated)
    #   mask black (0)   -> use background (source)
    #
    # We use the 512-resized source image (img_resized) and the 512-resized mask
    # (mask_512) to composite at the working resolution, then resize the result
    # back to the original image dimensions.
    #
    # IMPORTANT: result_img is already resized to (W_orig, H_orig) by the resize
    # call above. We composite at 512 first (before that resize) and then resize,
    # to keep the compositing in latent-aligned pixel space and avoid double-
    # interpolation artefacts on mask edges.

    # Composite at 512x512 before the final resize.
    # decoded_np is already uint8 (H=512, W=512, C=3) at this point.
    generated_512 = Image.fromarray(decoded_np)  # (512, 512) RGB

    # Build a 1-channel composite mask from mask_512.
    # mask_512 is uint8 (H, W) with 255=inpaint, 0=preserve.
    composite_mask_512 = Image.fromarray(mask_512, mode='L')

    # composite(foreground=generated, background=source, mask)
    composited_512 = Image.composite(generated_512, img_resized, composite_mask_512)

    # Now resize the composited result to original image dimensions.
    result_img = composited_512.resize((W_orig, H_orig), Image.LANCZOS)

    return result_img



print("run_inpainting defined.")


run_inpainting defined.


In [24]:
# -- §27.2  Inpainting smoke test (random inputs) ---------------------------
# Tests run_inpainting() with:
#   - a synthetic 512x512 image (gray)
#   - a random vlm_hidden tensor of correct shape
#   - a synthetic 50% coverage mask
# Verifies: output is PIL Image, correct size, no shape or dtype errors.

import torch
import numpy as np
from PIL import Image

assert vae is not None and unet is not None, "Load SD components first (§23)"
assert vlm_adapter is not None, "Load VLMProjectionAdapter (§23.2)"

smoke_img = Image.new('RGB', (512, 512), color=(128, 128, 128))

# Random VLM hidden vector (shape must match Phase 1 output).
_vlm_dim = 2048  # Qwen2.5-VL-3B-Instruct hidden size
smoke_vlm_h = torch.zeros(_vlm_dim, dtype=torch.float32)

# Mask: left half is edit region (255), right half is preserve (0).
smoke_mask = np.zeros((512, 512), dtype=np.uint8)
smoke_mask[:, :256] = 255

smoke_result = run_inpainting(
    source_image=smoke_img,
    vlm_hidden=smoke_vlm_h,
    instruction="make the left side blue",
    mask_np=smoke_mask,
    seed=42,
)

assert isinstance(smoke_result, Image.Image), f"Expected PIL Image, got {type(smoke_result)}"
assert smoke_result.size == smoke_img.size, (
    f"Output size {smoke_result.size} != input {smoke_img.size}"
)
print(f"Inpainting smoke test PASSED -- output size {smoke_result.size}")


Inpainting smoke test PASSED -- output size (512, 512)


/tmp/ipykernel_3220/227101825.py:239: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  composite_mask_512 = Image.fromarray(mask_512, mode='L')


## §28 — Full Inference Pipeline

`run_inference_pipeline(source_image, instruction)` chains all Phase 3 components:

1. `run_vlm_inference()` → edit_type, bbox_rel, edit_description  
2. `extract_hidden_state_inference()` → vlm_hidden (2048,) float32  
3. `bbox_to_sam2_mask()` → binary mask (H×W) uint8  
4. `run_inpainting()` → edited PIL Image  

Returns a metadata dict with all intermediate outputs for debugging and evaluation.

**Fail-fast policy:** each step asserts its output is valid before passing to the
next step. If VLM parsing fails (returns None) the fallback bbox is used; if
hidden-state extraction fails, a warning is issued and a zero vector is used.


In [25]:
# -- §28.1  run_inference_pipeline  (v3.0 — grounding-model branch) ---------
#
# Step ordering UNCHANGED:
#   1. VLM forward generation -> {edit_type, bbox, edit_description}.
#   2. VLM hidden-state extraction -> 2048-d conditioning vector.
#   3. Mask generation (v3.0: grounding -> SAM2 multi-mask -> rerank -> post).
#   4. SD inpainting with CFG, byte-identical to v2.2.
#
# What's new at the pipeline level:
#   * Step 3 receives the edit_description so the grounding model can phrase-prompt.
#   * Diagnostics dict from §26.1 is propagated into the metadata block, so
#     §29.1 can persist per-sample localization decisions for evaluation.

import time
import torch
from PIL import Image


def run_inference_pipeline(
    source_image: Image.Image,
    instruction: str,
    seed: int = None,
    device=None,
) -> dict:
    """Full Phase 3 inference pipeline (v3.0): VLM -> grounding -> SAM2 -> SD.

    Returns:
        dict with keys:
          edited_image  -- PIL Image (same size as source_image)
          vlm_result    -- dict from run_vlm_inference()
          mask_np       -- uint8 ndarray (H,W)
          mask_diag     -- per-sample mask diagnostics from §26.1
          vlm_hidden    -- Tensor (vlm_hidden_dim,) float32 CPU, or None
          elapsed_s     -- total wall-clock seconds
          metadata      -- dict with per-step timing and diagnostics
    """
    if device is None:
        device = DEVICE

    meta = {}
    t0 = time.time()

    # ── Step 1: VLM inference (still used for edit_type, JSON, bbox-as-fallback) ─
    t1 = time.time()
    vlm_result = run_vlm_inference(
        source_image=source_image,
        instruction=instruction,
        device=device,
    )
    meta["vlm_inference_s"] = time.time() - t1

    bbox_rel        = vlm_result["bbox"]
    edit_type       = vlm_result["edit_type"]
    edit_description = vlm_result.get("edit_description", instruction)
    used_fallback   = vlm_result.get("used_fallback", False)

    log.info(
        "Pipeline step 1: edit_type=%s bbox=%s desc=%r fallback=%s (%.2fs)",
        edit_type, bbox_rel, edit_description[:60], used_fallback,
        meta["vlm_inference_s"],
    )

    # ── Step 2: VLM hidden state extraction (UNCHANGED — Phase 2 conditioning) ──
    t2 = time.time()
    vlm_hidden = extract_hidden_state_inference(
        source_image=source_image,
        instruction=instruction,
        device=device,
    )
    meta["hidden_state_s"] = time.time() - t2

    if vlm_hidden is None:
        log.warning(
            "extract_hidden_state_inference returned None — "
            "falling back to zero conditioning vector."
        )
        vlm_hidden = torch.zeros(CFG.vlm_hidden_dim, dtype=torch.float32)
        meta["hidden_state_fallback"] = True
    else:
        meta["hidden_state_fallback"] = False

    log.info(
        "Pipeline step 2: vlm_hidden shape=%s (%.2fs)",
        tuple(vlm_hidden.shape), meta["hidden_state_s"],
    )

    # ── Step 3: v3.0 grounded mask generation ────────────────────────────
    t3 = time.time()
    mask_np, mask_diag = bbox_to_sam2_mask(
        source_image=source_image,
        bbox_rel=bbox_rel,
        edit_description=edit_description,
        instruction=instruction,
        device=device,
    )
    meta["sam2_mask_s"]    = time.time() - t3
    meta["mask_coverage"]  = mask_diag.get("final_mask_coverage")
    meta["mask_diag"]      = mask_diag

    log.info(
        "Pipeline step 3: mask shape=%s coverage=%.3f source=%s subject=%r (%.2fs)",
        mask_np.shape, meta["mask_coverage"], mask_diag["selected_source"],
        mask_diag["subject_phrase"], meta["sam2_mask_s"],
    )

    # ── Step 4: SD inpainting (UNCHANGED) ────────────────────────────────
    t4 = time.time()
    edited_image = run_inpainting(
        source_image=source_image,
        vlm_hidden=vlm_hidden,
        instruction=edit_description if edit_description else instruction,
        mask_np=mask_np,
        device=device,
        seed=seed,
    )
    meta["inpainting_s"] = time.time() - t4

    total_s = time.time() - t0
    log.info(
        "Pipeline complete in %.2fs "
        "(vlm=%.2f hs=%.2f mask=%.2f inp=%.2f)",
        total_s,
        meta["vlm_inference_s"],
        meta["hidden_state_s"],
        meta["sam2_mask_s"],
        meta["inpainting_s"],
    )

    return {
        "edited_image": edited_image,
        "vlm_result":   vlm_result,
        "mask_np":      mask_np,
        "mask_diag":    mask_diag,
        "vlm_hidden":   vlm_hidden,
        "elapsed_s":    total_s,
        "metadata":     meta,
    }


print("run_inference_pipeline defined (v3.0).")


run_inference_pipeline defined (v3.0).


In [26]:
# -- §28.2  End-to-end pipeline smoke test ----------------------------------
# Runs the full pipeline on a single manifest sample.
# This is the key integration test: all 4 steps must execute without error.

assert len(manifest_phase3) > 0, "No manifest samples -- run §20.2"

test_sample = manifest_phase3[0]
test_img_path = CFG.data_dir / test_sample['source_image']
test_instruction = test_sample.get('edit_instruction', 'edit this image')

print(f"Pipeline smoke test on: {test_sample['source_image']}")
print(f"Instruction: {test_instruction}")

test_src_img = Image.open(test_img_path).convert('RGB')

result = run_inference_pipeline(
    source_image=test_src_img,
    instruction=test_instruction,
    seed=42,
)

# Verify output structure.
assert isinstance(result['edited_image'], Image.Image), "edited_image not PIL"
assert result['edited_image'].size == test_src_img.size, "Size mismatch after pipeline"
assert result['mask_np'].dtype == np.uint8, "mask dtype wrong"
assert 'vlm_result' in result, "vlm_result missing"

print()
print("Pipeline smoke test PASSED.")
print(f"  edit_type     : {result['vlm_result']['edit_type']}")
print(f"  bbox (rel)    : {result['vlm_result']['bbox']}")
print(f"  mask coverage : {result['metadata']['mask_coverage']*100:.1f}%")
print(f"  elapsed (s)   : {result['elapsed_s']:.2f}")
print(f"  timing        : vlm={result['metadata']['vlm_inference_s']:.2f}  "
      f"hs={result['metadata']['hidden_state_s']:.2f}  "
      f"sam2={result['metadata']['sam2_mask_s']:.2f}  "
      f"inpaint={result['metadata']['inpainting_s']:.2f}")


Pipeline smoke test on: images/0000001_source.jpg
Instruction: Change vest positioned in the right-central area to red plaid

Pipeline smoke test PASSED.
  edit_type     : adjust
  bbox (rel)    : [489, 375, 569, 513]
  mask coverage : 1.1%
  elapsed (s)   : 9.16
  timing        : vlm=6.70  hs=0.43  sam2=0.51  inpaint=1.51


/tmp/ipykernel_3220/227101825.py:239: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  composite_mask_512 = Image.fromarray(mask_512, mode='L')


## §29 — Batch Inference on Manifest Samples

Runs `run_inference_pipeline()` on all Phase 3 manifest samples (or a configurable
subset). Saves results to `CFG.outputs_eval`:
- `<sample_id>_source.jpg` — original image
- `<sample_id>_edited.jpg` — pipeline output
- `<sample_id>_mask.png`   — SAM2 binary mask
- `inference_results.json` — metadata for all samples (bbox, edit_type, timing, etc.)

**Restart safety:** completed samples are detected by checking for the existence of
`<sample_id>_edited.jpg` before running. Re-running the cell resumes from where it
stopped.

**Error handling:** per-sample errors are caught and logged to `inference_results.json`
with `status: error`. The loop continues to the next sample.


In [27]:
# -- §29.1  Batch inference on manifest samples (v3.0) ----------------------
#
# Persists per-sample mask diagnostics (subject_phrase, selected_source,
# grounding score, SAM2 candidate scores, rerank pick, final coverage) so
# the v2.2-vs-v3.0 comparison in §30 can show *why* localization improved.

import json
import time
import traceback
from pathlib import Path
from PIL import Image

BATCH_LIMIT = 100   # set to e.g. 10 for a quick smoke run

output_dir = CFG.outputs_eval
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "inference_results.json"

if results_path.exists():
    with open(results_path) as f:
        all_results = json.load(f)
    completed_ids = {r["sample_id"] for r in all_results if r.get("status") == "ok"}
    print(f"Resuming: {len(completed_ids)} samples already completed.")
else:
    all_results = []
    completed_ids = set()

samples_to_run = manifest_phase3
if BATCH_LIMIT is not None:
    samples_to_run = samples_to_run[:BATCH_LIMIT]

print(f"Samples to process: {len(samples_to_run)} (limit={BATCH_LIMIT})")
print(f"Outputs dir: {output_dir}")

batch_t0 = time.time()

for idx, sample in enumerate(samples_to_run):
    sample_id = Path(sample["source_image"]).stem
    edited_path = output_dir / f"{sample_id}_edited.jpg"
    if sample_id in completed_ids and edited_path.exists():
        print(f"[{idx+1}/{len(samples_to_run)}] SKIP {sample_id} (already done)")
        continue

    img_path = CFG.data_dir / sample["source_image"]
    instruction = sample.get("edit_instruction", "edit this image")

    print(f"[{idx+1}/{len(samples_to_run)}] Processing {sample_id} ...")

    try:
        source_image = Image.open(img_path).convert("RGB")

        result = run_inference_pipeline(
            source_image=source_image,
            instruction=instruction,
            seed=42,
        )

        # Persist artifacts.
        source_image.save(output_dir / f"{sample_id}_source.jpg", quality=95)
        result["edited_image"].save(edited_path, quality=95)
        Image.fromarray(result["mask_np"]).save(
            output_dir / f"{sample_id}_mask.png"
        )

        vlm_r  = result["vlm_result"]
        diag   = result.get("mask_diag", {})
        record = {
            "sample_id":         sample_id,
            "source_image":      sample["source_image"],
            "edit_instruction":  instruction,
            "status":            "ok",
            "edit_type":         vlm_r["edit_type"],
            "bbox_rel":          vlm_r["bbox"],
            "edit_description":  vlm_r.get("edit_description", ""),
            "used_fallback":     vlm_r.get("used_fallback", False),
            "mask_coverage":     result["metadata"].get("mask_coverage"),
            "elapsed_s":         result["elapsed_s"],
            "timing": {
                k: v for k, v in result["metadata"].items()
                if isinstance(v, (int, float))
            },
            # v3.0: grounding-model diagnostics
            "subject_phrase":     diag.get("subject_phrase"),
            "selected_source":    diag.get("selected_source"),
            "grounding_n_boxes":  diag.get("grounding_n_boxes"),
            "grounding_top_score": diag.get("grounding_top_score"),
            "grounding_top_bbox_abs": diag.get("grounding_top_bbox_abs"),
            "selected_bbox_abs":  diag.get("selected_bbox_abs"),
            "vlm_bbox_degenerate": diag.get("vlm_bbox_degenerate"),
            "sam2_n_masks":       diag.get("sam2_n_masks"),
            "sam2_scores":        diag.get("sam2_scores"),
            "rerank_used":        diag.get("rerank_used"),
            "rerank_picked":      diag.get("rerank_picked"),
        }
        print(
            f"    OK  edit={vlm_r['edit_type']}  src={diag.get('selected_source')}  "
            f"subject={diag.get('subject_phrase')!r}  "
            f"coverage={(result['metadata'].get('mask_coverage') or 0)*100:.1f}%  "
            f"time={result['elapsed_s']:.1f}s"
        )

    except Exception as e:
        tb = traceback.format_exc()
        log.error("Sample %s failed: %s", sample_id, e)
        record = {
            "sample_id":         sample_id,
            "source_image":      sample["source_image"],
            "edit_instruction":  instruction,
            "status":            "error",
            "error":             str(e),
            "traceback":         tb,
        }
        print(f"    ERROR: {e}")

    all_results = [r for r in all_results if r["sample_id"] != sample_id]
    all_results.append(record)
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2)


n_ok  = sum(1 for r in all_results if r.get("status") == "ok")
n_err = sum(1 for r in all_results if r.get("status") == "error")
elapsed_total = time.time() - batch_t0

print()
print(f"Batch inference complete in {elapsed_total:.1f}s")
print(f"  Completed OK  : {n_ok}")
print(f"  Errors        : {n_err}")
print(f"  Results file  : {results_path}")


Samples to process: 100 (limit=100)
Outputs dir: /content/drive/MyDrive/img_edit_pipeline/outputs/eval_v3
[1/100] Processing 0000001_source ...


/tmp/ipykernel_3220/227101825.py:239: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  composite_mask_512 = Image.fromarray(mask_512, mode='L')


    OK  edit=adjust  src=grounding  subject='vest positioned in the right-central'  coverage=1.1%  time=9.2s
[2/100] Processing 0000002_source ...


    OK  edit=adjust  src=grounding  subject='bridge positioned in the upper-central'  coverage=13.8%  time=9.1s
[3/100] Processing 0000003_source ...


    OK  edit=adjust  src=grounding  subject='cliffs'  coverage=27.5%  time=9.0s
[4/100] Processing 0000004_source ...
    OK  edit=adjust  src=grounding  subject='flowers positioned in the upper-central'  coverage=1.7%  time=8.9s
[5/100] Processing 0000005_source ...
    OK  edit=adjust  src=grounding  subject='pillows positioned in the top-left'  coverage=11.1%  time=8.8s
[6/100] Processing 0000007_source ...


...

    OK  edit=adjust  src=grounding  subject='muscle car positioned in the'  coverage=35.0%  time=9.0s
[75/100] Processing 0000088_source ...
    OK  edit=adjust  src=grounding  subject='car'  coverage=7.9%  time=8.5s
[76/100] Processing 0000090_source ...


    OK  edit=unknown  src=grounding  subject='pile of crushed cars'  coverage=2.7%  time=9.1s
[77/100] Processing 0000091_source ...
    OK  edit=adjust  src=grounding  subject='dress positioned in the central-right'  coverage=11.5%  time=8.8s
[78/100] Processing 0000092_source ...
    OK  edit=adjust  src=grounding  subject='wedding dress positioned in the'  coverage=4.4%  time=9.0s
[79/100] Processing 0000093_source ...
    OK  edit=adjust  src=grounding  subject='side doors positioned in the'  coverage=5.3%  time=9.0s
[80/100] Processing 0000094_source ...


    OK  edit=unknown  src=grounding  subject='grass positioned in the lower'  coverage=29.3%  time=9.1s
[81/100] Processing 0000096_source ...
    OK  edit=adjust  src=grounding  subject='pancakes'  coverage=9.1%  time=8.7s
[82/100] Processing 0000097_source ...
    OK  edit=adjust  src=grounding  subject='head'  coverage=29.6%  time=8.6s
[83/100] Processing 0000098_source ...


    OK  edit=adjust  src=grounding  subject='trees'  coverage=5.5%  time=9.0s
[84/100] Processing 0000099_source ...


    OK  edit=unknown  src=grounding  subject='fallen leaves positioned in the'  coverage=6.4%  time=9.2s
[85/100] Processing 0000100_source ...
    OK  edit=adjust  src=grounding  subject='flag'  coverage=5.4%  time=8.8s
[86/100] Processing 0000101_source ...
    OK  edit=adjust  src=grounding  subject='pancakes positioned in the top-right'  coverage=25.6%  time=9.0s
[87/100] Processing 0000102_source ...
    OK  edit=adjust  src=grounding  subject='jacket positioned in the central-right'  coverage=24.0%  time=8.8s
[88/100] Processing 0000104_source ...
    OK  edit=adjust  src=grounding  subject='flowers positioned in the central-right'  coverage=1.9%  time=8.8s
[89/100] Processing 0000107_source ...
    OK  edit=adjust  src=grounding  subject='couple'  coverage=5.7%  time=9.4s
[90/100] Processing 0000109_source ...
    OK  edit=adjust  src=grounding  subject='box of teddy grahams positioned'  coverage=17.8%  time=9.2s
[91/100] Processing 0000110_source ...
    OK  edit=adjust  src=gr

## §30 — Results Summary

Loads `inference_results.json` and prints a summary table: per-edit-type breakdown,
average timing, fallback rate, and mask coverage stats. Run after §29.


In [28]:
# -- §30.1  Results summary (v3.0) ------------------------------------------

import json
from pathlib import Path
from collections import defaultdict, Counter

results_path = CFG.outputs_eval / "inference_results.json"
assert results_path.exists(), f"No results file found at {results_path}"

with open(results_path) as f:
    results = json.load(f)

ok_results  = [r for r in results if r.get("status") == "ok"]
err_results = [r for r in results if r.get("status") == "error"]

print(f"Total samples  : {len(results)}")
print(f"OK             : {len(ok_results)}")
print(f"Errors         : {len(err_results)}")
print()

if ok_results:
    by_type = defaultdict(list)
    for r in ok_results:
        by_type[r.get("edit_type", "unknown")].append(r)

    print(f"{'Edit type':<15} {'Count':>6} {'FallbackRate':>14} "
          f"{'AvgCoverage%':>14} {'AvgTime(s)':>12}")
    print("-" * 65)
    for et in sorted(by_type):
        recs = by_type[et]
        fb_rate = sum(1 for r in recs if r.get("used_fallback")) / len(recs)
        avg_cov = sum((r.get("mask_coverage") or 0) for r in recs) / len(recs)
        avg_t   = sum(r.get("elapsed_s", 0) for r in recs) / len(recs)
        print(f"{et:<15} {len(recs):>6} {fb_rate*100:>13.1f}% "
              f"{avg_cov*100:>13.1f}% {avg_t:>12.2f}")

    print()

    # v3.0 — grounding diagnostics
    src_counter = Counter(r.get("selected_source") for r in ok_results)
    print("Localization source distribution (v3.0):")
    for src, n in src_counter.most_common():
        print(f"  {src!s:<14} {n:>5}  ({n/len(ok_results)*100:.1f}%)")

    g_n = [r.get("grounding_n_boxes") or 0 for r in ok_results]
    g_top = [r.get("grounding_top_score") for r in ok_results
             if r.get("grounding_top_score") is not None]
    print(f"Grounding boxes per sample: mean={sum(g_n)/max(len(g_n),1):.2f} "
          f"max={max(g_n) if g_n else 0}")
    if g_top:
        print(f"Grounding top score: mean={sum(g_top)/len(g_top):.3f} "
              f"min={min(g_top):.3f} max={max(g_top):.3f}")

    rerank_used_n = sum(1 for r in ok_results if r.get("rerank_used"))
    print(f"CLIP rerank fired on {rerank_used_n} / {len(ok_results)} samples "
          f"({rerank_used_n/len(ok_results)*100:.1f}%)")

    deg_n = sum(1 for r in ok_results if r.get("vlm_bbox_degenerate"))
    print(f"VLM bbox judged degenerate on {deg_n} / {len(ok_results)} samples "
          f"({deg_n/len(ok_results)*100:.1f}%)  -- these were the v2.2 failure cases")

    print()
    all_elapsed = [r.get("elapsed_s", 0) for r in ok_results]
    fallback_total = sum(1 for r in ok_results if r.get("used_fallback"))
    print(f"Overall avg time  : {sum(all_elapsed)/len(all_elapsed):.2f}s")
    print(f"VLM fallback rate : {fallback_total/len(ok_results)*100:.1f}%")

if err_results:
    print()
    print("Error samples:")
    for r in err_results:
        print(f"  {r['sample_id']}: {r.get('error', '?')}")


Total samples  : 100
OK             : 100
Errors         : 0

Edit type        Count   FallbackRate   AvgCoverage%   AvgTime(s)
-----------------------------------------------------------------
adjust              92           0.0%          14.6%         8.91
unknown              8         100.0%          10.3%         8.65

Localization source distribution (v3.0):
  grounding         96  (96.0%)
  global_edit        4  (4.0%)
Grounding boxes per sample: mean=2.32 max=5
Grounding top score: mean=0.592 min=0.289 max=0.980
CLIP rerank fired on 96 / 100 samples (96.0%)
VLM bbox judged degenerate on 13 / 100 samples (13.0%)  -- these were the v2.2 failure cases

Overall avg time  : 8.89s
VLM fallback rate : 8.0%
